In [1]:
!pip install rasterio geopandas shapely pyproj numpy requests
!pip install pymupdf
!pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 105.9 MB/s eta 0:00:00


In [20]:
# ============================================================
# CONFIGURATION & CONSTANTS
# ============================================================
from pathlib import Path
import requests

# ============================================================
# Population Helpers
# ============================================================

NOMINATIM_URL = "https://nominatim.openstreetmap.org"
WORLDPOP_API_URL = "https://api.worldpop.org/v2"

HEADERS = {
    "User-Agent": "Hackathon/1.0",
    "Accept": "application/json",
}

HOUSEHOLD_FILE = Path("./data_cache/un_household_2026.xlsx")
HOUSEHOLD_DOWNLOAD_URL = "https://population.un.org/household/assets/UNDESA_PD_2026_hh-size-composition.xlsx"
DATA_MAX_AGE_DAYS = 30

# ============================================================
# Competetor Helper
# ============================================================
OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

# ============================================================
# MArket Price Helper
# ============================================================

AGMARKNET_BASE_URL = "https://api.agmarknet.gov.in/v1"
AGMARKNET_SESSION = requests.Session()
AGMARKNET_SESSION.headers.update({
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://agmarknet.gov.in",
    "Referer": "https://agmarknet.gov.in/",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    ),
})

DEFAULT_INDIAN_STATES = {
    "andaman and nicobar": 1, "andhra pradesh": 2, "arunachal pradesh": 3,
    "assam": 4, "bihar": 5, "chandigarh": 6, "chhattisgarh": 7,
    "dadra and nagar haveli": 8, "daman and diu": 9, "delhi": 10,
    "goa": 11, "gujarat": 12, "haryana": 13, "himachal pradesh": 14,
    "jammu and kashmir": 15, "jharkhand": 16, "karnataka": 17, "kerala": 18,
    "ladakh": 38, "lakshadweep": 19, "madhya pradesh": 20, "maharashtra": 21,
    "manipur": 22, "meghalaya": 23, "mizoram": 24, "nagaland": 25,
    "odisha": 26, "puducherry": 27, "punjab": 28, "rajasthan": 29,
    "sikkim": 30, "tamil nadu": 31, "telangana": 37, "tripura": 32,
    "uttar pradesh": 34, "uttarakhand": 33, "west bengal": 36
}


# ============================================================
# Supply Chain Helper
# ============================================================
NOMINATIM_URL = "https://nominatim.openstreetmap.org"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSRM_URL = "http://router.project-osrm.org/route/v1/driving"

# ============================================================
# Transportation Helper
# ============================================================

NOMINATIM_URL = "https://nominatim.openstreetmap.org"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSRM_URL = "http://router.project-osrm.org/route/v1/driving"


# ============================================================
# Seasonal Helper
# ============================================================
NOMINATIM_URL = "https://nominatim.openstreetmap.org"
OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
OSRM_URL = "http://router.project-osrm.org/route/v1/driving"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]




GEMINI_MODEL_NAME = "gemini-3.5-flash-lite"

In [3]:
import json
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pycountry
import requests
from shapely.geometry import Point


# ============================================================
# 1. GEOCODING & SPATIAL BUFFER
# ============================================================

def geocode_location(location: str) -> tuple[float, float]:
    """Geocode an address string to (latitude, longitude)."""
    params = {
        "q": location,
        "format": "json",
        "limit": 1,
    }
    response = requests.get(
        f"{NOMINATIM_URL}/search",
        params=params,
        headers=HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    results = response.json()

    if not results:
        raise ValueError(f"Location not found: {location}")

    return float(results[0]["lat"]), float(results[0]["lon"])


def get_country_from_coordinates(latitude: float, longitude: float) -> dict:
    """Reverse geocode coordinates to obtain country name, ISO2, and ISO3 codes."""
    params = {
        "lat": latitude,
        "lon": longitude,
        "format": "jsonv2",
        "zoom": 3,
        "addressdetails": 1,
    }
    response = requests.get(
        f"{NOMINATIM_URL}/reverse",
        params=params,
        headers=HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()

    address = data.get("address", {})
    country = address.get("country")
    iso2 = address.get("country_code", "").upper()

    if not country or not iso2:
        raise RuntimeError(f"Could not resolve country for: {latitude}, {longitude}")

    country_obj = pycountry.countries.get(alpha_2=iso2)
    if country_obj is None:
        raise LookupError(f"Unknown ISO2 country code: {iso2}")

    return {
        "country": country,
        "country_code": iso2,
        "country_code_iso3": country_obj.alpha_3,
    }


def create_radius_geojson(
    latitude: float,
    longitude: float,
    radius_km: float,
) -> dict:
    """Generate a circular polygon buffer in meters via UTM and return GeoJSON geometry."""
    point = gpd.GeoDataFrame(
        geometry=[Point(longitude, latitude)],
        crs="EPSG:4326",
    )

    zone = int((longitude + 180) / 6) + 1
    utm_epsg = (32600 if latitude >= 0 else 32700) + zone

    point_utm = point.to_crs(f"EPSG:{utm_epsg}")
    buffer_utm = point_utm.geometry.iloc[0].buffer(radius_km * 1000)

    buffer_wgs84 = gpd.GeoSeries(
        [buffer_utm],
        crs=f"EPSG:{utm_epsg}",
    ).to_crs("EPSG:4326")

    return buffer_wgs84.iloc[0].__geo_interface__


# ============================================================
# 2. WORLDPOP API INTEGRATION
# ============================================================

def submit_worldpop_task(
    geojson: dict,
    year: int = 2025,
    resolution: str = "100m",
    max_retries: int = 5,
) -> dict:
    """Submit a calculation task to WorldPop."""
    payload = {
        "geojson": geojson,
        "year": year,
        "resolution": resolution,
    }
    url = f"{WORLDPOP_API_URL}/population"
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Submitting WorldPop task (attempt {attempt}/{max_retries})...")
            response = requests.post(
                url,
                json=payload,
                headers=HEADERS,
                timeout=(30, 180),
            )
            print(f"WorldPop HTTP status: {response.status_code}")
            response.raise_for_status()

            data = response.json()
            if "task_id" not in data:
                raise RuntimeError(f"WorldPop returned unexpected response: {data}")
            return data

        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            last_error = e
            print(f"WorldPop connection error: {e}")
            if attempt < max_retries:
                wait = min(2 ** attempt, 20)
                print(f"Retrying in {wait}s...")
                time.sleep(wait)

    raise RuntimeError(
        f"Failed to connect to WorldPop after {max_retries} attempts."
    ) from last_error


def wait_for_worldpop_result(
    task_id: str,
    timeout_seconds: int = 300,
    poll_interval: int = 3,
) -> dict:
    """Poll WorldPop task status until finished."""
    url = f"{WORLDPOP_API_URL}/tasks/{task_id}"
    start_time = time.time()

    while True:
        if time.time() - start_time > timeout_seconds:
            raise TimeoutError("WorldPop calculation timed out.")

        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            data = response.json()
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            print(f"Polling connection issue: {e}")
            time.sleep(poll_interval)
            continue

        status = data.get("status")
        print(f"WorldPop status: {status}")

        if status == "success":
            return data.get("result", {})
        if status == "failure":
            raise RuntimeError(f"WorldPop calculation failed: {data}")

        time.sleep(poll_interval)


# ============================================================
# 3. UN HOUSEHOLD DATA HANDLERS
# ============================================================

def download_un_household_data():
    """Download the UN DESA household size Excel sheet."""
    HOUSEHOLD_FILE.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading UN household dataset...")

    response = requests.get(HOUSEHOLD_DOWNLOAD_URL, headers=HEADERS, timeout=120)
    response.raise_for_status()

    if not response.content:
        raise RuntimeError("UN household dataset returned empty content.")

    with open(HOUSEHOLD_FILE, "wb") as f:
        f.write(response.content)
    print(f"Dataset saved to: {HOUSEHOLD_FILE}")


def household_data_needs_refresh() -> bool:
    """Check if file exists and is under the max cache age threshold."""
    if not HOUSEHOLD_FILE.exists():
        return True

    modified_time = datetime.fromtimestamp(
        HOUSEHOLD_FILE.stat().st_mtime, tz=timezone.utc
    )
    return (datetime.now(timezone.utc) - modified_time) > timedelta(
        days=DATA_MAX_AGE_DAYS
    )


def load_un_household_data() -> pd.DataFrame:
    """Read the UN Excel file and locate the header row."""
    if household_data_needs_refresh():
        download_un_household_data()
    else:
        modified_time = datetime.fromtimestamp(
            HOUSEHOLD_FILE.stat().st_mtime, tz=timezone.utc
        )
        age_days = (datetime.now(timezone.utc) - modified_time).days
        print(f"Using cached UN dataset ({age_days} days old).")

    sheet = "HH size and composition 2026"
    raw = pd.read_excel(
        HOUSEHOLD_FILE, sheet_name=sheet, header=None, engine="openpyxl"
    )

    header_row = None
    for i in range(min(30, len(raw))):
        row_text = " ".join(raw.iloc[i].astype(str).str.lower().str.strip().tolist())
        if (
            "country and area" in row_text
            and "average household size" in row_text
        ):
            header_row = i
            break

    if header_row is None:
        raise RuntimeError("Could not locate UN household data header.")

    df = pd.read_excel(
        HOUSEHOLD_FILE,
        sheet_name=sheet,
        header=header_row,
        engine="openpyxl",
    ).dropna(axis=1, how="all").dropna(axis=0, how="all")

    df.columns = [
        str(c).strip().replace("\n", " ").replace("\r", " ")
        for c in df.columns
    ]
    return df


def get_average_household_size(country_code_iso3: str) -> float:
    """Retrieve the average household size for an ISO3 country code."""
    df = load_un_household_data()
    iso_col = "ISO3 Code"
    hh_col = "Average household size (number of members)"

    if iso_col not in df.columns or hh_col not in df.columns:
        raise RuntimeError(f"Required columns missing from UN data: {iso_col}, {hh_col}")

    iso_series = df[iso_col].astype(str).str.strip().str.upper()
    matches = df[iso_series == country_code_iso3.upper()]

    if matches.empty:
        raise LookupError(f"No UN household data for ISO3: {country_code_iso3}")

    values = pd.to_numeric(matches[hh_col], errors="coerce").dropna()
    values = values[values > 0]

    if values.empty:
        raise LookupError(f"Missing UN household size value for ISO3: {country_code_iso3}")

    return float(values.iloc[0])


# ============================================================
# 4. MASTER ANALYSIS PIPELINE
# ============================================================

def analyze_market_reach(
    location: str,
    radius_km: float,
    year: int = 2025,
) -> dict:
    """End-to-end analysis: geocoding, population, and household estimates."""
    print(f"1. Geocoding location: {location}")
    latitude, longitude = geocode_location(location)
    print(f"   Coordinates: {latitude}, {longitude}")

    print(f"2. Querying country metadata...")
    country_info = get_country_from_coordinates(latitude, longitude)
    iso3 = country_info["country_code_iso3"]
    print(f"   Found {country_info['country']} ({iso3})")

    print(f"3. Building {radius_km} km radius buffer...")
    geojson = create_radius_geojson(latitude, longitude, radius_km)

    print("4. Calculating population via WorldPop...")
    task = submit_worldpop_task(geojson, year=year)
    wp_result = wait_for_worldpop_result(task["task_id"])

    population = round(wp_result["total_population"])

    print("5. Looking up UN average household size...")
    household_size = get_average_household_size(iso3)
    estimated_households = round(population / household_size)

    return {
        "location": location,
        "latitude": latitude,
        "longitude": longitude,
        "radius_km": radius_km,
        "country": country_info["country"],
        "country_code": country_info["country_code"],
        "country_code_iso3": iso3,
        "population": population,
        "area_km2": wp_result["area_km2"],
        "population_density": wp_result["population_density"],
        "data_year": wp_result["data_year"],
        "population_data_source": wp_result["data_source"],
        "average_household_size": round(household_size, 3),
        "estimated_households": estimated_households,
        "household_estimation_method": "population / average household size",
        "household_data_source": "UN DESA Household Size and Composition 2026",
    }


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    result = analyze_market_reach(
        location="Katwa, Purba Bardhaman, West Bengal",

        # location="kento, japan",
        radius_km=3.28,
        year=2025,
    )
    print("\n--- FINAL OUTPUT ---")
    print(json.dumps(result, indent=4, ensure_ascii=False))

1. Geocoding location: Katwa, Purba Bardhaman, West Bengal
   Coordinates: 23.644278, 88.1287333
2. Querying country metadata...
   Found India (IND)
3. Building 3.28 km radius buffer...
4. Calculating population via WorldPop...
Submitting WorldPop task (attempt 1/5)...
WorldPop HTTP status: 200
WorldPop status: success
5. Looking up UN average household size...
Dataset saved to: data_cache/un_household_2026.xlsx

--- FINAL OUTPUT ---
{
    "location": "Katwa, Purba Bardhaman, West Bengal",
    "latitude": 23.644278,
    "longitude": 88.1287333,
    "radius_km": 3.28,
    "country": "India",
    "country_code": "IN",
    "country_code_iso3": "IND",
    "population": 137660,
    "area_km2": 33.7602,
    "population_density": 4077.59,
    "data_year": 2025,
    "population_data_source": "worldpop_R2025A_2025_100m",
    "average_household_size": 5.09,
    "estimated_households": 27045,
    "household_estimation_method": "population / average household size",
    "household_data_source": "

In [4]:
import os
import math
import json
import re
import time
import requests
from typing import List, Dict, Optional, Tuple
from google import genai
from google.genai import types

try:
    from google.colab import userdata
except ImportError:
    userdata = None

# ============================================================
# CONFIGURATION
# ============================================================

COMPETETOR_CACHE_FILE_PATH = "gemini_taxonomy_cache.json"

# ============================================================
# CACHING HELPERS
# ============================================================

def load_cache() -> Dict[str, Dict]:
    """Loads the taxonomy cache from disk."""
    if os.path.exists(COMPETETOR_CACHE_FILE_PATH):
        try:
            with open(COMPETETOR_CACHE_FILE_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(cache: Dict[str, Dict]) -> None:
    """Saves the taxonomy cache to disk."""
    try:
        with open(COMPETETOR_CACHE_FILE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"[Warning] Failed to write cache to file: {e}")

# ============================================================
# DISTANCE & STRING UTILITIES
# ============================================================

def haversine_distance_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0088
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))

def get_bounding_box(lat: float, lon: float, radius_km: float) -> Tuple[float, float, float, float]:
    """Generates an indexed bounding box (south, west, north, east) for Overpass."""
    delta_lat = radius_km / 111.0
    delta_lon = radius_km / (111.0 * math.cos(math.radians(lat)))
    return (
        lat - delta_lat,
        lon - delta_lon,
        lat + delta_lat,
        lon + delta_lon
    )

def normalize_text(value: Optional[str]) -> str:
    if not value:
        return ""
    value = str(value).lower().strip()
    return re.sub(r"\s+", " ", value)

def normalize_business_name(name: Optional[str]) -> str:
    if not name:
        return ""
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

# ============================================================
# 1. FEW-SHOT GEMINI TAG & KEYWORD GENERATOR (WITH CACHING)
# ============================================================

FEW_SHOT_SYSTEM_INSTRUCTION = """
You are an expert OpenStreetMap (OSM) GIS Engineer and multilingual rural retail taxonomist.
Given any target business description or trade, generate a structured JSON profile:
1. `primary_osm_tags`: High-confidence exact OSM key-value pairs (e.g., [["shop", "dairy"]]).
2. `broad_osm_tags`: General contextual fallback tags where rural unclassified shops are often filed (e.g. grocery, convenience, supermarket, farm, general).
3. `keywords`: Comprehensive lowercase aliases, product names, local vernacular terms (especially Indic/South Asian words like doodh, dudh, kisan, krishi, aahar, etc.), and prominent regional brand names.

--- EXAMPLE 1 ---
Input Business: "dairy"
Output:
{
  "primary_osm_tags": [["shop", "dairy"]],
  "broad_osm_tags": [["shop", "grocery"], ["shop", "supermarket"], ["shop", "convenience"], ["shop", "general"], ["landuse", "farm"], ["amenity", "marketplace"]],
  "keywords": ["dairy", "milk", "milk shop", "milk centre", "milk center", "milk parlour", "milk parlor", "milk booth", "milk point", "milk store", "milk depot", "milk supplier", "milk delivery", "dudh", "doodh", "dugdha", "dugdho", "dudh ghar", "dudh ghor", "amul", "mother dairy", "heritage fresh", "nandini", "saras", "verka", "sudha", "paneer", "curd", "ghee"]
}

--- EXAMPLE 2 ---
Input Business: "agricultural inputs and seeds"
Output:
{
  "primary_osm_tags": [["shop", "agrarian"], ["shop", "seeds"], ["shop", "fertilizer"], ["shop", "farm"]],
  "broad_osm_tags": [["shop", "grocery"], ["shop", "general"], ["shop", "hardware"], ["landuse", "farm"]],
  "keywords": ["krishi", "kisan", "agro", "agri", "beej", "seeds", "fertilizer", "pesticide", "khad", "kisan kendra", "krishi seva", "agrochemical", "urea", "dap", "iffco", "kribhco", "bayer", "syngenta"]
}

--- EXAMPLE 3 ---
Input Business: "poultry and chicken shop"
Output:
{
  "primary_osm_tags": [["shop", "butcher"], ["shop", "seafood"]],
  "broad_osm_tags": [["shop", "grocery"], ["shop", "general"], ["landuse", "farm"], ["amenity", "marketplace"]],
  "keywords": ["poultry", "chicken", "meat", "mutton", "broiler", "egg", "egg shop", "murgi", "gosht", "mangsho", "hatchery", "desi chicken", "suguna", "venky", "shanthi feeds"]
}
"""

def generate_business_profile_with_llm(
    business_type: str,
    api_key: Optional[str] = None
) -> Dict:
    normalized_key = business_type.strip().lower()

    # 1. Check local cache first
    cache = load_cache()
    if normalized_key in cache:
        print(f"\n[AI Taxonomy Cache] Loaded taxonomy for '{business_type}' from cache.")
        return cache[normalized_key]

    # 2. Cache miss -> Call Gemini
    print(f"\n[AI Taxonomy] Querying Gemini for: '{business_type}'...")
    key = api_key
    if not key and userdata:
        try:
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass
    if not key:
        key = os.environ.get("GEMINI_API_KEY")

    client = genai.Client(api_key=key)
    prompt = f'Target Business: "{business_type}"\nGenerate the JSON matching the required schema.'

    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=FEW_SHOT_SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            temperature=0.1,
        ),
    )

    try:
        profile = json.loads(response.text)
        print(f"[AI Taxonomy] Generated {len(profile.get('primary_osm_tags', []))} primary tags and {len(profile.get('keywords', []))} keywords.")

        # 3. Store to cache file
        cache[normalized_key] = profile
        save_cache(cache)
        return profile
    except Exception as e:
        raise RuntimeError(f"Failed to parse LLM response as JSON: {response.text}") from e

# ============================================================
# 2. OVERPASS QUERY BUILDER & CLASSIFIER
# ============================================================

def build_overpass_query(
    latitude: float,
    longitude: float,
    radius_km: float,
    profile: Dict,
) -> str:
    """
    Builds an Overpass query using indexed bounding box tags.
    """
    s, w, n, e = get_bounding_box(latitude, longitude, radius_km)
    bbox = f"{s:.6f},{w:.6f},{n:.6f},{e:.6f}"
    clauses = []

    all_tags = profile.get("primary_osm_tags", []) + profile.get("broad_osm_tags", [])
    seen_tags = set()

    for item in all_tags:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            key, val = item[0], item[1]
            pair = (key, val)
            if pair not in seen_tags:
                seen_tags.add(pair)
                clauses.append(f'nwr["{key}"="{val}"]({bbox});')

    if not clauses:
        clauses.append(f'nwr["shop"]({bbox});')

    clauses_block = "\n    ".join(clauses)
    return f"""[out:json][timeout:30];
(
    {clauses_block}
);
out center tags;
"""

def is_candidate_match(tags: Dict, profile: Dict) -> Tuple[bool, Optional[str]]:
    """Filters Overpass results in Python memory."""
    primary_tags = profile.get("primary_osm_tags", [])
    broad_tags = profile.get("broad_osm_tags", [])
    keywords = [k.lower().strip() for k in profile.get("keywords", []) if len(k.strip()) >= 2]

    # Check exact primary tag match
    for item in primary_tags:
        if len(item) == 2 and tags.get(item[0]) == item[1]:
            return True, f"primary_tag:{item[0]}={item[1]}"

    # Searchable tag texts
    searchable_fields = [
        tags.get("name"),
        tags.get("brand"),
        tags.get("operator"),
        tags.get("description"),
        tags.get("product"),
        tags.get("produce"),
        tags.get("animal"),
        tags.get("livestock"),
    ]
    combined_text = normalize_text(" ".join(f for f in searchable_fields if f))

    if not combined_text:
        return False, None

    for kw in keywords:
        pattern = r"\b" + re.escape(kw) + r"\b"
        if re.search(pattern, combined_text):
            return True, f"keyword:{kw}"

    for item in broad_tags:
        if len(item) == 2 and tags.get(item[0]) == item[1]:
            for kw in keywords:
                if kw in combined_text:
                    return True, f"{item[0]}={item[1]}+keyword:{kw}"

    return False, None

def get_osm_coordinates(element: Dict) -> Tuple[Optional[float], Optional[float]]:
    if "lat" in element and "lon" in element:
        return float(element["lat"]), float(element["lon"])
    center = element.get("center")
    if center and "lat" in center and "lon" in center:
        return float(center["lat"]), float(center["lon"])
    return None, None

# ============================================================
# 3. OVERPASS RETRIEVAL ENGINE
# ============================================================

def fetch_osm_competitors(
    latitude: float,
    longitude: float,
    radius_km: float,
    profile: Dict,
    business_label: str
) -> List[Dict]:
    query = build_overpass_query(latitude, longitude, radius_km, profile)

    last_error = None
    for server in OVERPASS_SERVERS:
        try:
            print(f"Connecting to Overpass server: {server}")

            response = requests.post(
                OVERPASS_URL,
                data={"data": query},
                headers=HEADERS,
                timeout=45,
            )

            if response.status_code != 200:
                print(f"Warning: {server} responded with HTTP {response.status_code}")
                time.sleep(1)
                continue

            data = response.json()
            elements = data.get("elements", [])
            print(f"Server returned {len(elements)} raw candidate features. Filtering locally in Python...")

            competitors = []
            seen_osm_ids = set()

            for element in elements:
                osm_type = element.get("type")
                osm_id = element.get("id")
                uid = (osm_type, osm_id)

                if uid in seen_osm_ids:
                    continue
                seen_osm_ids.add(uid)

                tags = element.get("tags", {})
                is_candidate, reason = is_candidate_match(tags, profile)
                if not is_candidate:
                    continue

                lat, lon = get_osm_coordinates(element)
                if lat is None or lon is None:
                    continue

                dist = haversine_distance_km(latitude, longitude, lat, lon)
                if dist > radius_km:
                    continue

                competitors.append({
                    "source": "OpenStreetMap",
                    "osm_id": osm_id,
                    "osm_type": osm_type,
                    "name": tags.get("name") or "Unnamed Establishment",
                    "brand": tags.get("brand"),
                    "operator": tags.get("operator"),
                    "business_type": business_label,
                    "latitude": lat,
                    "longitude": lon,
                    "distance_km": round(dist, 3),
                    "match_reason": reason,
                    "tags": tags,
                })

            competitors.sort(key=lambda x: x["distance_km"])
            print(f"Verified {len(competitors)} valid competitors for '{business_label}'")
            return competitors

        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"Warning: {server} failed: {e}")
            last_error = e
            time.sleep(1)

            raise RuntimeError(f"All Overpass mirrors failed. Last error: {last_error}")

# ============================================================
# 4. DEDUPLICATION & METRICS
# ============================================================

def deduplicate_competitors(
    competitors: List[Dict],
    distance_threshold_m: float = 40.0,
) -> List[Dict]:
    final = []
    for comp in competitors:
        lat = comp["latitude"]
        lon = comp["longitude"]
        name = normalize_business_name(comp.get("name"))

        duplicate = False
        for existing in final:
            dist_m = haversine_distance_km(
                lat, lon, existing["latitude"], existing["longitude"]
            ) * 1000

            if dist_m <= distance_threshold_m:
                duplicate = True
                break

            existing_name = normalize_business_name(existing.get("name"))
            if (
                name
                and existing_name
                and name != "unnamed establishment"
                and name == existing_name
                and dist_m <= 200.0
            ):
                duplicate = True
                break

        if not duplicate:
            final.append(comp)
    return final

def calculate_distance_buckets(competitors: List[Dict], radius_km: float) -> Dict:
    return {
        "within_2km": sum(1 for x in competitors if x["distance_km"] <= 2.0),
        "within_5km": sum(1 for x in competitors if x["distance_km"] <= 5.0),
        "within_10km": sum(1 for x in competitors if x["distance_km"] <= 10.0),
    }

# ============================================================
# 5. MASTER ANALYSIS PIPELINE
# ============================================================

def analyze_competitors(
    latitude: float,
    longitude: float,
    population: int,
    business_type: str,
    radius_km: float = 10.0,
    gemini_api_key: Optional[str] = None,
) -> Dict:
    print("==========================================")
    print("AI-POWERED OSM COMPETITOR PIPELINE")
    print("==========================================")
    print(f"Target Query : '{business_type}'")
    print(f"Center Point : {latitude}, {longitude}")
    print(f"Radius       : {radius_km} km")

    # Cached AI lookup
    profile = generate_business_profile_with_llm(
        business_type=business_type,
        api_key=gemini_api_key
    )

    # Overpass Query
    raw_competitors = fetch_osm_competitors(
        latitude=latitude,
        longitude=longitude,
        radius_km=radius_km,
        profile=profile,
        business_label=business_type,
    )

    unique_competitors = deduplicate_competitors(raw_competitors)
    buckets = calculate_distance_buckets(unique_competitors, radius_km)
    pop_k = (population / 1000) if (population and population > 0) else 0

    density_metrics = {
        "2km": round(buckets["within_2km"] / pop_k, 4) if pop_k > 0 else 0.0,
        "5km": round(buckets["within_5km"] / pop_k, 4) if pop_k > 0 else 0.0,
        "10km": round(buckets["within_10km"] / pop_k, 4) if pop_k > 0 else 0.0,
    }

    return {
        "search": {
            "latitude": latitude,
            "longitude": longitude,
            "radius_km": radius_km,
            "business_type": business_type,
        },
        "competitor_summary": {
            "total_unique_competitors": len(unique_competitors),
            "within_2km": buckets["within_2km"],
            "within_5km": buckets["within_5km"],
            "within_10km": buckets["within_10km"],
            "competitor_density_per_1000_people": density_metrics,
        },
        "sources": {
            "openstreetmap": len(unique_competitors),
        },
        "competitors": unique_competitors,
    }

# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    LATITUDE = 22.5726459
    LONGITUDE = 88.3638953
    POPULATION = 5509424

    api_key = None
    if userdata:
        try:
            api_key = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass
    if not api_key:
        api_key = os.environ.get("GEMINI_API_KEY")

    result = analyze_competitors(
        latitude=LATITUDE,
        longitude=LONGITUDE,
        population=POPULATION,
        business_type="goat farming and live animal breeding",
        radius_km=10.0,
        gemini_api_key=api_key,
    )

    print("\n--- FINAL OUTPUT ---")
    print(json.dumps(result, indent=4, ensure_ascii=False))

AI-POWERED OSM COMPETITOR PIPELINE
Target Query : 'goat farming and live animal breeding'
Center Point : 22.5726459, 88.3638953
Radius       : 10.0 km

[AI Taxonomy] Querying Gemini for: 'goat farming and live animal breeding'...
[AI Taxonomy] Generated 3 primary tags and 17 keywords.
Connecting to Overpass server: https://overpass-api.de/api/interpreter
Server returned 63 raw candidate features. Filtering locally in Python...
Verified 1 valid competitors for 'goat farming and live animal breeding'

--- FINAL OUTPUT ---
{
    "search": {
        "latitude": 22.5726459,
        "longitude": 88.3638953,
        "radius_km": 10.0,
        "business_type": "goat farming and live animal breeding"
    },
    "competitor_summary": {
        "total_unique_competitors": 1,
        "within_2km": 0,
        "within_5km": 1,
        "within_10km": 1,
        "competitor_density_per_1000_people": {
            "2km": 0.0,
            "5km": 0.0002,
            "10km": 0.0002
        }
    },
    "s

In [5]:
import os
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional

# Google GenAI SDK & Colab Userdata
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False


# ============================================================
# CONFIGURATION & CACHE
# ============================================================

PRICE_CACHE_DIR = Path("market_price_data")
PRICE_CACHE_DIR.mkdir(exist_ok=True)
CACHE_MAX_AGE_SECONDS = 24 * 60 * 60


def get_api_key() -> str:
    """Fetches Gemini API key from Colab userdata or environment variables."""
    if COLAB_AVAILABLE:
        try:
            key = userdata.get("GEMINI_API_KEY")
            if key:
                return key
        except Exception:
            pass
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        raise ValueError(
            "Gemini API key not found. Add 'GEMINI_API_KEY' to Colab Secrets (🔑) "
            "or set it as an environment variable."
        )
    return key


# ============================================================
# 1. AI PROFILE & COMMODITY FALLBACK GENERATOR
# ============================================================

LLM_SYSTEM_PROMPT = """
You are an expert Indian Agricultural and Market Commodity Taxonomist specializing in AGMARKNET and Mandi trading systems.

Given ANY user business trade or sector:
Generate a JSON object with:
1. `sources`: Subset of ["AGMARKNET_PRICES", "AGMARKNET_QUANTITIES", "AGMARKNET_HISTORICAL", "DOCA_RETAIL", "DOCA_WHOLESALE"]
2. `commodities`: An array of trade commodities. Each item must have:
   - `standard_name`: Standard English commodity name.
   - `agmarknet_probable_id`: Probable or common numeric Agmarknet ID if known, else null.
   - `aliases`: Common mandi aliases, vernacular terms (Hindi/Bengali/etc.), and exact Agmarknet display names (e.g., "Goat", "Live Goat", "Meat", "Paddy(Dhan)(Common)", "Potato").
"""

def generate_business_price_profile(business_type: str) -> Dict[str, Any]:
    print(f"\n[AI Profiler] Generating Agmarknet mappings for '{business_type}'...")
    client = genai.Client(api_key=get_api_key())

    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=f'Target Business: "{business_type}"\nOutput JSON matching the schema.',
        config=types.GenerateContentConfig(
            system_instruction=LLM_SYSTEM_PROMPT,
            response_mime_type="application/json",
            temperature=0.1,
        ),
    )

    try:
        profile = json.loads(response.text)
        print(f"[AI Profiler] Mapped {len(profile.get('commodities', []))} commodities.")
        return profile
    except Exception as e:
        raise RuntimeError(f"Failed to parse Gemini output: {response.text}") from e


# ============================================================
# 2. UTILITY & CACHING HELPERS
# ============================================================

def cache_is_fresh(path: Path, max_age_seconds: int = CACHE_MAX_AGE_SECONDS) -> bool:
    return path.exists() and (time.time() - path.stat().st_mtime) < max_age_seconds

def normalize_name(value: Any) -> str:
    if value is None:
        return ""
    return " ".join(str(value).strip().lower().replace("_", " ").replace("-", " ").split())

def recursive_find_lists(obj: Any) -> List[Dict[str, Any]]:
    """Recursively traverses any arbitrary nested JSON structure to locate dict lists."""
    results = []
    if isinstance(obj, dict):
        for val in obj.values():
            if isinstance(val, list):
                for item in val:
                    if isinstance(item, dict):
                        results.append(item)
            results.extend(recursive_find_lists(val))
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                results.append(item)
            results.extend(recursive_find_lists(item))
    return results


# ============================================================
# 3. AGMARKNET ENDPOINTS & SAFE RETRIEVAL
# ============================================================

def agmarknet_get(path: str, params: Optional[Dict[str, Any]] = None) -> Optional[Any]:
    url = f"{AGMARKNET_BASE_URL}{path}"
    try:
        response = AGMARKNET_SESSION.get(url, params=params, timeout=30)
        print(f"AGMARKNET GET {path} -> HTTP {response.status_code}")
        if response.status_code != 200:
            return None
        content_type = response.headers.get("content-type", "")
        if "json" not in content_type.lower():
            return None
        return response.json()
    except Exception as e:
        print(f"AGMARKNET request failed on {path}: {e}")
        return None

def load_agmarknet_states(force: bool = False) -> List[Dict[str, Any]]:
    cache_file = PRICE_CACHE_DIR / "agmarknet_states.json"
    if not force and cache_is_fresh(cache_file):
        return json.loads(cache_file.read_text(encoding="utf-8"))

    print("Fetching AGMARKNET states...")
    all_states: List[Dict[str, Any]] = []

    for page in range(1, 10):
        data = agmarknet_get("/location/state", params={"page": page})
        if not data:
            break
        records = [item for item in recursive_find_lists(data) if any(k in str(item).lower() for k in ["state", "statename"])]
        if not records:
            break
        all_states.extend(records)

    if all_states:
        cache_file.write_text(json.dumps(all_states, ensure_ascii=False), encoding="utf-8")
    return all_states

def resolve_state_id(state: str) -> int:
    requested = normalize_name(state)

    # 1. Check local lookup table first
    if requested in DEFAULT_INDIAN_STATES:
        return DEFAULT_INDIAN_STATES[requested]

    # 2. Check Agmarknet live states
    states_data = load_agmarknet_states()
    for record in states_data:
        text = json.dumps(record).lower()
        if requested in text:
            for key, value in record.items():
                if ("id" in key.lower() or "state" in key.lower()) and str(value).isdigit():
                    return int(value)

    raise RuntimeError(f"Could not resolve AGMARKNET state ID for: '{state}'")


# ============================================================
# 4. COMMODITY RESOLUTION WITH MULTI-SOURCE FALLBACK
# ============================================================

def load_agmarknet_commodities(force: bool = False) -> List[Dict[str, Any]]:
    cache_file = PRICE_CACHE_DIR / "agmarknet_commodities.json"
    if not force and cache_is_fresh(cache_file):
        return json.loads(cache_file.read_text(encoding="utf-8"))

    print("Downloading AGMARKNET commodity catalogue...")
    candidates = []

    # Attempt standard Agmarknet commodity endpoints
    for endpoint in ["/commodity", "/commodity/all", "/daily-price-arrival/filters", "/location/commodity"]:
        data = agmarknet_get(endpoint)
        if data:
            flattened = recursive_find_lists(data)
            for item in flattened:
                # Check for commodity-like fields
                keys = [k.lower() for k in item.keys()]
                if any("commodity" in k or "crop" in k for k in keys):
                    candidates.append(item)
            if candidates:
                break

    # Deduplicate
    unique = []
    seen = set()
    for item in candidates:
        key = json.dumps(item, sort_keys=True, default=str)
        if key not in seen:
            seen.add(key)
            unique.append(item)

    if unique:
        cache_file.write_text(json.dumps(unique, ensure_ascii=False), encoding="utf-8")
        print(f"Loaded {len(unique)} commodities from Agmarknet API.")
    else:
        print("Note: Agmarknet filters did not return a commodity table; using direct search fallback.")

    return unique

def get_commodity_name(record: Dict[str, Any]) -> str:
    for key in ["commodityName", "commodity_name", "commodity", "name", "label", "commodityDesc"]:
        if key in record and record[key]:
            return str(record[key])
    return ""

def get_commodity_id(record: Dict[str, Any]) -> Optional[int]:
    for key in ["commodityId", "commodity_id", "id", "value", "commodityCode"]:
        if key in record and str(record[key]).isdigit():
            return int(record[key])
    return None

def resolve_commodity(commodity_obj: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Resolves commodity name and ID via catalogue fuzzy match or AI guidance."""
    std_name = commodity_obj.get("standard_name", "")
    aliases = [normalize_name(x) for x in commodity_obj.get("aliases", [])]
    if std_name:
        aliases.insert(0, normalize_name(std_name))

    records = load_agmarknet_commodities()

    if records:
        scored = []
        for record in records:
            cname = normalize_name(get_commodity_name(record))
            if not cname:
                continue

            score = 0
            if any(cname == a for a in aliases):
                score = 100
            elif any(a in cname for a in aliases):
                score = 85
            elif any(cname in a for a in aliases):
                score = 70

            if score > 0:
                cid = get_commodity_id(record)
                if cid is not None:
                    scored.append((score, cname, cid, record))

        if scored:
            scored.sort(key=lambda x: (-x[0], x[1]))
            score, match_name, cid, record = scored[0]
            print(f"Matched '{std_name}' -> Agmarknet '{match_name}' (ID: {cid})")
            return {"id": cid, "name": match_name, "score": score}

    # Fallback to AI-provided ID if Agmarknet's catalog lookup endpoint is blank
    probable_id = commodity_obj.get("agmarknet_probable_id")
    if probable_id:
        return {"id": int(probable_id), "name": std_name, "score": 60}

    return None


# ============================================================
# 5. DATA EXTRACTION & STATISTICS
# ============================================================

def flatten_json_records(obj: Any) -> List[Dict[str, Any]]:
    records = []
    if isinstance(obj, dict):
        if any(k.lower() in {"minprice", "maxprice", "modalprice", "modal_price", "price", "commodityprice"} for k in obj.keys()):
            records.append(obj)
        for value in obj.values():
            records.extend(flatten_json_records(value))
    elif isinstance(obj, list):
        for item in obj:
            records.extend(flatten_json_records(item))
    return records

def extract_price_values(data: Any) -> List[float]:
    prices = []
    for record in flatten_json_records(data):
        for key, value in record.items():
            key_norm = normalize_name(key)
            if any(x in key_norm for x in ["min price", "max price", "modal price", "modal_price", "price"]):
                try:
                    number = float(str(value).replace(",", "").replace("₹", "").strip())
                    if 0 < number < 300000:
                        prices.append(number)
                except (ValueError, TypeError):
                    pass
    return prices

def calculate_price_statistics(prices: List[float]) -> Dict[str, Any]:
    if not prices:
        return {"data_points": 0, "min": None, "median": None, "mean": None, "max": None}
    prices = sorted(prices)
    n = len(prices)
    median = prices[n // 2] if n % 2 else (prices[n // 2 - 1] + prices[n // 2]) / 2
    return {
        "data_points": n,
        "min": round(min(prices), 2),
        "median": round(median, 2),
        "mean": round(sum(prices) / n, 2),
        "max": round(max(prices), 2),
    }


# ============================================================
# 6. MASTER PIPELINE
# ============================================================

def analyze_market_price(business_type: str, state: str = "West Bengal") -> Dict[str, Any]:
    print("=" * 60)
    print("AI-POWERED AGMARKNET MARKET PRICE ANALYSIS")
    print("=" * 60)
    print(f"Business : {business_type}")
    print(f"State    : {state}")
    print("=" * 60)

    profile = generate_business_price_profile(business_type)
    state_id = resolve_state_id(state)

    result = {
        "business_type": business_type,
        "state": state,
        "state_id": state_id,
        "ai_generated_profile": profile,
        "sources": {},
    }

    today = datetime.now()
    commodities = profile.get("commodities", [])

    # 1. State Daily Prices
    result["sources"]["agmarknet_prices"] = {}
    for comm in commodities:
        std_name = comm.get("standard_name")
        resolved = resolve_commodity(comm)

        daily_data = agmarknet_get(
            "/prices-and-arrivals/commodity-market/daily-report-state",
            params={"date": today.strftime("%Y-%m-%d"), "state": state_id, "includeExcel": "false"},
        )

        if daily_data:
            prices = extract_price_values(daily_data)
            result["sources"]["agmarknet_prices"][std_name] = {
                "matched_commodity": resolved["name"] if resolved else std_name,
                "commodity_id": resolved["id"] if resolved else None,
                "statistics": calculate_price_statistics(prices),
                "data_status": "received",
            }
        else:
            result["sources"]["agmarknet_prices"][std_name] = {
                "status": "No data returned for today from Agmarknet."
            }

    # 2. Historical Prices
    if "AGMARKNET_HISTORICAL" in profile.get("sources", []):
        result["sources"]["agmarknet_historical"] = {}
        for comm in commodities:
            std_name = comm.get("standard_name")
            resolved = resolve_commodity(comm)
            if not resolved or not resolved.get("id"):
                continue

            history = []
            for i in range(3):
                hist_date = today - timedelta(days=30 * i)
                data = agmarknet_get(
                    "/prices-and-arrivals/date-wise/specific-commodity",
                    params={
                        "year": hist_date.year,
                        "month": hist_date.month,
                        "stateId": state_id,
                        "commodityId": resolved["id"],
                        "includeExcel": "false",
                    },
                )
                prices = extract_price_values(data) if data else []
                history.append({
                    "year": hist_date.year,
                    "month": hist_date.month,
                    "statistics": calculate_price_statistics(prices),
                })
            result["sources"]["agmarknet_historical"][std_name] = history

    return result


# ============================================================
# RUN (TEST ON ARBITRARY BUSINESS)
# ============================================================

if __name__ == "__main__":
    SAMPLE_BUSINESS = "goat farming and mutton trade"
    TARGET_STATE = "West Bengal"

    output = analyze_market_price(business_type=SAMPLE_BUSINESS, state=TARGET_STATE)

    print("\n--- FINAL OUTPUT (PREVIEW) ---")
    print(json.dumps({
        "business_type": output["business_type"],
        "state": output["state"],
        "mapped_commodities": [c["standard_name"] for c in output["ai_generated_profile"].get("commodities", [])],
        "prices_retrieved": output["sources"].get("agmarknet_prices", {})
    }, indent=2))

AI-POWERED AGMARKNET MARKET PRICE ANALYSIS
Business : goat farming and mutton trade
State    : West Bengal

[AI Profiler] Generating Agmarknet mappings for 'goat farming and mutton trade'...
[AI Profiler] Mapped 3 commodities.
AGMARKNET GET /commodity -> HTTP 404
AGMARKNET GET /commodity/all -> HTTP 404
AGMARKNET GET /daily-price-arrival/filters -> HTTP 200
AGMARKNET GET /location/commodity -> HTTP 404
Note: Agmarknet filters did not return a commodity table; using direct search fallback.
AGMARKNET GET /prices-and-arrivals/commodity-market/daily-report-state -> HTTP 200
AGMARKNET GET /commodity -> HTTP 404
AGMARKNET GET /commodity/all -> HTTP 404
AGMARKNET GET /daily-price-arrival/filters -> HTTP 200
AGMARKNET GET /location/commodity -> HTTP 404
Note: Agmarknet filters did not return a commodity table; using direct search fallback.
AGMARKNET GET /prices-and-arrivals/commodity-market/daily-report-state -> HTTP 200
AGMARKNET GET /commodity -> HTTP 404
AGMARKNET GET /commodity/all -> HTTP

In [6]:
import os
import math
import requests
import json
from typing import Any, Dict, List, Tuple, Optional

from google import genai
from google.genai import types

try:
    from google.colab import userdata
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False

def get_gemini_client() -> genai.Client:
    key = None
    if COLAB_AVAILABLE:
        try:
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass
    key = key or os.environ.get("GEMINI_API_KEY")
    if not key:
        raise ValueError("Missing 'GEMINI_API_KEY'. Set it in Colab Secrets or as an env variable.")
    return genai.Client(api_key=key)


def get_coordinates(location_name: str) -> Tuple[float, float]:
    params = {"q": location_name, "format": "json", "limit": 1}
    res = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=15)
    res.raise_for_status()
    data = res.json()
    if not data:
        raise ValueError(f"Could not geocode location: '{location_name}'")
    return float(data[0]["lat"]), float(data[0]["lon"])


def get_supply_chain_profile(business_type: str) -> Dict[str, Any]:
    """Dynamically generates industry-specific supply chain pillars via Gemini."""
    client = get_gemini_client()
    prompt = f"""
    You are an expert in rural/semi-urban supply chain networks.
    Analyze the business type: "{business_type}".
    Define exactly 3 to 4 critical supply chain pillars (e.g., raw materials, specialist services, marketplace, logistics).

    Return a strictly valid JSON object:
    {{
      "pillars": [
        {{
          "id": "raw_materials",
          "label": "Raw Materials & Inputs",
          "weight": 0.35,
          "ideal_km": 5.0,
          "cutoff_km": 25.0,
          "osm_queries": ["nwr['shop'~'agrarian|fertilizer']", "nwr['commercial'='agricultural']"]
        }}
      ]
    }}
    Weights must sum to 1.0. Use realistic Overpass QL tag clauses inside `osm_queries`.
    """

    resp = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1
        )
    )
    return json.loads(resp.text)


def haversine_distance_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    return round(2 * r * math.asin(math.sqrt(a)), 2)


def get_road_distance_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    try:
        url = f"{OSRM_URL}/{lon1},{lat1};{lon2},{lat2}?overview=false"
        res = requests.get(url, headers=HEADERS, timeout=5)
        if res.status_code == 200:
            routes = res.json().get("routes", [])
            if routes:
                return round(routes[0]["distance"] / 1000.0, 2)
    except Exception:
        pass
    return round(haversine_distance_km(lat1, lon1, lat2, lon2) * 1.25, 2)


def calculate_score(dist_km: float, ideal_km: float, cutoff_km: float) -> int:
    if dist_km <= ideal_km:
        return int(100 - (dist_km / ideal_km) * 10)
    elif dist_km >= cutoff_km:
        return 10
    decay = (dist_km - ideal_km) / (cutoff_km - ideal_km)
    return max(10, int(90 - decay * 80))


def evaluate_supply_chain(location_name: str, business_type: str) -> Dict[str, Any]:
    lat, lon = get_coordinates(location_name)
    profile = get_supply_chain_profile(business_type)
    radius_meters = 25000

    results = {}
    pillar_scores = {}
    weighted_sum = 0.0

    for pillar in profile.get("pillars", []):
        pid = pillar["id"]
        label = pillar["label"]
        weight = pillar["weight"]
        ideal_km = pillar.get("ideal_km", 5.0)
        cutoff_km = pillar.get("cutoff_km", 25.0)

        # Build union query for this specific pillar
        clause_str = "\n".join(f"  {q}(around:{radius_meters},{lat},{lon});" for q in pillar["osm_queries"])
        overpass_q = f"[out:json][timeout:25];\n(\n{clause_str}\n);\nout center 15;"

        elements = []
        try:
            res = requests.post(OVERPASS_URL, data={"data": overpass_q}, headers=HEADERS, timeout=30)
            if res.status_code == 200:
                elements = res.json().get("elements", [])
        except Exception:
            pass

        pois = []
        for el in elements:
            p_lat = el.get("lat") or el.get("center", {}).get("lat")
            p_lon = el.get("lon") or el.get("center", {}).get("lon")
            if not p_lat or not p_lon:
                continue
            tags = el.get("tags", {})
            name = tags.get("name") or tags.get("operator") or f"Local {label} Facility"
            pois.append({"name": name, "lat": p_lat, "lon": p_lon})

        if pois:
            nearest = min(pois, key=lambda p: haversine_distance_km(lat, lon, p["lat"], p["lon"]))
            dist_km = get_road_distance_km(lat, lon, nearest["lat"], nearest["lon"])
            score = calculate_score(dist_km, ideal_km, cutoff_km)
            results[pid] = {
                "pillar": label,
                "status": "Found",
                "nearest_name": nearest["name"],
                "distance_km": dist_km,
                "score": score
            }
        else:
            score = 15
            results[pid] = {
                "pillar": label,
                "status": "Unmapped / Not Found within 25km",
                "nearest_name": None,
                "distance_km": None,
                "score": score
            }

        pillar_scores[pid] = score
        weighted_sum += score * weight

    return {
        "location": location_name,
        "coordinates": {"latitude": lat, "longitude": lon},
        "business_type": business_type,
        "pillars": results,
        "overall_supply_chain_score": round(weighted_sum, 1)
    }


if __name__ == "__main__":
    # Example: Works for carpentry, pottery, or any non-livestock venture
    result = evaluate_supply_chain("Madhyamgram, Kolkata, West Bengal", business_type="wooden furniture manufacturing")
    print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "location": "Madhyamgram, Kolkata, West Bengal",
  "coordinates": {
    "latitude": 22.6947839,
    "longitude": 88.4530183
  },
  "business_type": "wooden furniture manufacturing",
  "pillars": {
    "raw_materials": {
      "pillar": "Raw Materials & Timber Sourcing",
      "status": "Unmapped / Not Found within 25km",
      "nearest_name": null,
      "distance_km": null,
      "score": 15
    },
    "specialist_services": {
      "pillar": "Hardware, Tools & Finishing Supplies",
      "status": "Unmapped / Not Found within 25km",
      "nearest_name": null,
      "distance_km": null,
      "score": 15
    },
    "logistics": {
      "pillar": "Freight, Transport & Distribution",
      "status": "Found",
      "nearest_name": "Bengal Timber",
      "distance_km": 3.85,
      "score": 97
    },
    "marketplace": {
      "pillar": "Retail Showrooms & B2B Buyers",
      "status": "Found",
      "nearest_name": "Bengal Timber",
      "distance_km": 3.85,
      "score": 98
    }
  }

In [16]:
import os
import math
import json
import requests
from typing import Any, Dict, Tuple

from google import genai
from google.genai import types

try:
    from google.colab import userdata
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False

# ============================================================
# GEMINI CLIENT
# ============================================================

def get_gemini_client() -> genai.Client:
    key = None

    if COLAB_AVAILABLE:
        try:
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass

    key = key or os.environ.get("GEMINI_API_KEY")
    key = key or os.environ.get("GOOGLE_API_KEY")

    if not key:
        raise ValueError(
            "Missing Gemini API key. Set GEMINI_API_KEY "
            "in Colab Secrets or as an environment variable."
        )

    return genai.Client(api_key=key)


# ============================================================
# 1. GEOCODING & ROUTING
# ============================================================

def get_coordinates(location_name: str) -> Tuple[float, float]:
    params = {
        "q": location_name,
        "format": "json",
        "limit": 1
    }

    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=HEADERS,
        timeout=15
    )

    response.raise_for_status()

    data = response.json()

    if not data:
        raise ValueError(
            f"Could not resolve coordinates for '{location_name}'"
        )

    return (
        float(data[0]["lat"]),
        float(data[0]["lon"])
    )


def haversine_distance_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float
) -> float:
    radius = 6371.0

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )

    return round(
        2 * radius * math.asin(math.sqrt(a)),
        2
    )


def calculate_road_route(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float
) -> Dict[str, Any]:

    straight_line_km = haversine_distance_km(
        lat1,
        lon1,
        lat2,
        lon2
    )

    try:
        url = (
            f"{OSRM_URL}/"
            f"{lon1},{lat1};{lon2},{lat2}"
            "?overview=false"
        )

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        )

        if response.status_code == 200:
            routes = response.json().get("routes", [])

            if routes:
                road_km = round(
                    routes[0]["distance"] / 1000.0,
                    2
                )

                duration_min = round(
                    routes[0]["duration"] / 60.0,
                    1
                )

                detour = round(
                    road_km / straight_line_km,
                    2
                ) if straight_line_km > 0 else 1.0

                return {
                    "straight_line_km": straight_line_km,
                    "road_distance_km": road_km,
                    "duration_minutes": duration_min,
                    "detour_factor": detour,
                    "routing_engine": "OSRM Live Network"
                }

    except Exception as exc:
        print(
            f"[ROUTING] OSRM failed: {exc}"
        )

    # Fallback approximation.
    estimated_road_km = round(
        straight_line_km * 1.30,
        2
    )

    estimated_duration = round(
        (estimated_road_km / 25.0) * 60,
        1
    )

    return {
        "straight_line_km": straight_line_km,
        "road_distance_km": estimated_road_km,
        "duration_minutes": estimated_duration,
        "detour_factor": 1.30,
        "routing_engine": (
            "Haversine Rural Approximation (1.30x)"
        )
    }


# ============================================================
# COORDINATE NORMALIZATION
# ============================================================

def normalize_coordinates(
    coordinates: Any
) -> Tuple[float, float]:
    """
    Accept all of these formats:

    {"lat": 23.64, "lon": 88.12}

    {"latitude": 23.64, "longitude": 88.12}

    (23.64, 88.12)

    [23.64, 88.12]

    Returns:
        (latitude, longitude)
    """

    if isinstance(coordinates, dict):
        lat = (
            coordinates.get("lat")
            if coordinates.get("lat") is not None
            else coordinates.get("latitude")
        )

        lon = (
            coordinates.get("lon")
            if coordinates.get("lon") is not None
            else coordinates.get("longitude")
        )

        if lat is None or lon is None:
            raise ValueError(
                f"Invalid coordinate dictionary: {coordinates}"
            )

        return float(lat), float(lon)

    if isinstance(coordinates, (tuple, list)):
        if len(coordinates) < 2:
            raise ValueError(
                f"Invalid coordinate tuple/list: {coordinates}"
            )

        return (
            float(coordinates[0]),
            float(coordinates[1])
        )

    raise TypeError(
        f"Unsupported coordinate type: "
        f"{type(coordinates).__name__}: {coordinates}"
    )


# ============================================================
# 2. SUPPLY CHAIN PROFILE
# ============================================================

def generate_supply_chain_profile(
    business_type: str
) -> Dict[str, Any]:

    client = get_gemini_client()

    prompt = f"""
You are an expert rural supply chain architect
specializing in emerging economies and India.

Target Business:
"{business_type}"

Define 3 to 4 critical upstream/downstream
supply-chain facility pillars.

OpenStreetMap tags in rural areas are often broad
and non-standard.

Use both specific and broad fallback tags.

For example:

Veterinary:
- amenity=veterinary
- healthcare=veterinary
- shop=chemist
- amenity=pharmacy

Feed/input:
- shop=agrarian
- shop=animal_feed
- shop=general
- shop=convenience
- commercial=agricultural

Meat/processing:
- shop=butcher
- amenity=marketplace
- landuse=commercial

Markets/logistics:
- amenity=marketplace
- highway=bus_stop
- amenity=bus_station
- railway=station

Return ONLY valid JSON:

{{
  "pillars": [
    {{
      "id": "veterinary_care",
      "label": "Veterinary & Animal Care",
      "weight": 0.25,
      "ideal_km": 8.0,
      "cutoff_km": 30.0,
      "osm_queries": [
        "nwr['amenity'='veterinary']",
        "nwr['healthcare'='veterinary']",
        "nwr['shop'='chemist']"
      ]
    }}
  ]
}}

Weights MUST sum to 1.0.
"""

    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1
        )
    )

    result = json.loads(response.text)

    if not isinstance(result, dict):
        raise ValueError(
            "Gemini supply-chain profile was not a JSON object."
        )

    pillars = result.get("pillars")

    if not isinstance(pillars, list):
        raise ValueError(
            "Gemini supply-chain profile does not contain "
            "a valid 'pillars' list."
        )

    return result


# ============================================================
# 3. OVERPASS DISCOVERY
# ============================================================

def discover_supply_chain_nodes(
    lat: float,
    lon: float,
    profile: Dict[str, Any],
    radius_meters: int = 30000
) -> Dict[str, Any]:

    discovered_pillars = {}

    for pillar in profile.get("pillars", []):

        pid = pillar.get("id", "unknown")
        label = pillar.get(
            "label",
            pid
        )

        print(
            f"Searching Overpass for '{label}' "
            f"within {radius_meters / 1000:.1f} km..."
        )

        queries = pillar.get(
            "osm_queries",
            []
        )

        if not queries:
            print(
                f"[OVERPASS] No queries for {pid}"
            )

        clause_str = "\n".join(
            f"  {query}(around:{radius_meters},{lat},{lon});"
            for query in queries
        )

        query = f"""
[out:json][timeout:30];
(
{clause_str}
);
out center 20;
"""

        elements = []

        try:
            response = requests.post(
                OVERPASS_URL,
                data={"data": query},
                headers=HEADERS,
                timeout=35
            )

            if response.status_code == 200:
                elements = response.json().get(
                    "elements",
                    []
                )
            else:
                print(
                    f"[OVERPASS] HTTP {response.status_code}"
                )

        except Exception as exc:
            print(
                f"[OVERPASS] Query failed for {pid}: {exc}"
            )

        candidates = []

        for element in elements:

            element_lat = element.get("lat")
            element_lon = element.get("lon")

            center = element.get(
                "center",
                {}
            )

            if element_lat is None:
                element_lat = center.get("lat")

            if element_lon is None:
                element_lon = center.get("lon")

            if element_lat is None or element_lon is None:
                continue

            tags = element.get(
                "tags",
                {}
            )

            name = (
                tags.get("name")
                or tags.get("name:en")
                or tags.get("operator")
                or tags.get("brand")
                or (
                    f"Regional {label} Node "
                    f"({tags.get('shop') or tags.get('amenity') or 'Hub'})"
                )
            )

            candidates.append({
                "name": name,
                "lat": float(element_lat),
                "lon": float(element_lon),
                "tags": tags
            })

        if candidates:

            nearest = min(
                candidates,
                key=lambda item: haversine_distance_km(
                    lat,
                    lon,
                    item["lat"],
                    item["lon"]
                )
            )

            distance = haversine_distance_km(
                lat,
                lon,
                nearest["lat"],
                nearest["lon"]
            )

            discovered_pillars[pid] = {
                "pillar_label": label,
                "status": "Found",
                "name": nearest["name"],
                "coordinates": {
                    "lat": nearest["lat"],
                    "lon": nearest["lon"]
                },
                "straight_distance_km": distance,
                "weight": float(
                    pillar.get("weight", 0.0)
                ),
                "ideal_km": float(
                    pillar.get("ideal_km", 8.0)
                ),
                "cutoff_km": float(
                    pillar.get("cutoff_km", 35.0)
                )
            }

        else:

            # IMPORTANT:
            # This is NOT a real facility.
            # It is only a synthetic analytical anchor.

            fallback_distance = round(
                float(
                    pillar.get(
                        "ideal_km",
                        10.0
                    )
                ) * 1.4,
                2
            )

            fallback_lat = lat + 0.08
            fallback_lon = lon + 0.06

            discovered_pillars[pid] = {
                "pillar_label": label,
                "status": (
                    "Unmapped Locally "
                    "(Sub-Divisional HQ Estimated)"
                ),
                "name": (
                    f"Estimated Sub-Divisional "
                    f"{label} Center"
                ),
                "coordinates": {
                    "lat": fallback_lat,
                    "lon": fallback_lon
                },
                "straight_distance_km": fallback_distance,
                "weight": float(
                    pillar.get("weight", 0.0)
                ),
                "ideal_km": float(
                    pillar.get("ideal_km", 8.0)
                ),
                "cutoff_km": float(
                    pillar.get("cutoff_km", 35.0)
                )
            }

    return discovered_pillars


# ============================================================
# 4. DYNAMIC FREIGHT PROFILING
# ============================================================

def generate_dynamic_freight_parameters(
    business_type: str,
    origin_name: str,
    destinations: Dict[str, Dict[str, Any]],
    orig_lat: float,
    orig_lon: float
) -> Dict[str, Any]:

    client = get_gemini_client()

    summary = {}

    for key, destination in destinations.items():

        if not isinstance(destination, dict):
            raise TypeError(
                f"Destination '{key}' must be a dictionary, "
                f"got {type(destination).__name__}"
            )

        if "coordinates" not in destination:
            raise ValueError(
                f"Destination '{key}' has no coordinates."
            )

        # ====================================================
        # FIX FOR YOUR ERROR
        # ====================================================
        #
        # coordinates may be:
        #
        # {"lat": ..., "lon": ...}
        #
        # OR:
        #
        # (lat, lon)
        #
        # normalize_coordinates() handles both.
        # ====================================================

        dest_lat, dest_lon = normalize_coordinates(
            destination["coordinates"]
        )

        distance_km = destination.get(
            "straight_distance_km"
        )

        if distance_km is None:
            distance_km = haversine_distance_km(
                orig_lat,
                orig_lon,
                dest_lat,
                dest_lon
            )

        destination["straight_distance_km"] = distance_km

        summary[key] = {
            "facility_name": destination.get(
                "name",
                "Unknown Facility"
            ),
            "distance_km": distance_km
        }

    prompt = f"""
You are a commercial freight cost estimator
for rural and peri-urban India.

Business Type:
"{business_type}"

Origin:
"{origin_name}"

Target Facilities:
{json.dumps(summary, ensure_ascii=False)}

For each node key return:

1. recommended_vehicle
2. payload_capacity_kg
3. base_fare_inr
4. base_km
5. rate_per_km_inr
6. round_trip_multiplier
7. monthly_trips
8. handling_notes

Return ONLY JSON:

{{
  "freight_specifications": {{
    "<node_key>": {{
      "recommended_vehicle": "Tata Ace",
      "payload_capacity_kg": 1000,
      "base_fare_inr": 200.0,
      "base_km": 4.0,
      "rate_per_km_inr": 24.0,
      "round_trip_multiplier": 1.8,
      "monthly_trips": 4,
      "handling_notes": "Bulk bags / crates"
    }}
  }}
}}
"""

    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1
        )
    )

    try:
        result = json.loads(response.text)

        if not isinstance(result, dict):
            raise ValueError(
                "Freight response is not a JSON object."
            )

        specifications = result.get(
            "freight_specifications",
            {}
        )

        if not isinstance(specifications, dict):
            raise ValueError(
                "'freight_specifications' is not a dictionary."
            )

        return specifications

    except Exception as exc:
        raise RuntimeError(
            f"Failed to parse logistics parameters: "
            f"{response.text}"
        ) from exc


# ============================================================
# 5. TRIP COST
# ============================================================

def compute_trip_cost(
    distance_km: float,
    spec: Dict[str, Any]
) -> Dict[str, float]:

    base_fare = float(
        spec.get("base_fare_inr", 180.0)
    )

    base_km = float(
        spec.get("base_km", 4.0)
    )

    rate_km = float(
        spec.get("rate_per_km_inr", 22.0)
    )

    round_trip_multiplier = float(
        spec.get(
            "round_trip_multiplier",
            1.8
        )
    )

    if distance_km <= base_km:
        one_way = base_fare
    else:
        one_way = (
            base_fare
            + (
                distance_km - base_km
            ) * rate_km
        )

    return {
        "one_way_inr": round(
            one_way,
            2
        ),
        "round_trip_inr": round(
            one_way * round_trip_multiplier,
            2
        )
    }


# ============================================================
# 6. MASTER PIPELINE
# ============================================================

def run_supply_chain_and_transportation_analysis(
    business_type: str,
    origin_location: str,
    search_radius_meters: int = 30000
) -> Dict[str, Any]:

    print("=" * 65)
    print(
        "INTEGRATED SUPPLY CHAIN & "
        "TRANSPORTATION OPEX PIPELINE"
    )
    print("=" * 65)

    print(
        f"Business Profile : {business_type}"
    )

    print(
        f"Origin Location  : {origin_location}"
    )

    # --------------------------------------------------------
    # ORIGIN
    # --------------------------------------------------------

    orig_lat, orig_lon = get_coordinates(
        origin_location
    )

    print(
        f"Origin Resolved  : "
        f"{orig_lat:.4f}, {orig_lon:.4f}"
    )

    # --------------------------------------------------------
    # STAGE 1
    # --------------------------------------------------------

    print(
        "\n[Stage 1] "
        "Formulating supply chain requirements..."
    )

    profile = generate_supply_chain_profile(
        business_type
    )

    # --------------------------------------------------------
    # STAGE 2
    # --------------------------------------------------------

    print(
        "\n[Stage 2] "
        "Scanning OpenStreetMap for physical "
        "supply chain hubs..."
    )

    discovered_nodes = (
        discover_supply_chain_nodes(
            orig_lat,
            orig_lon,
            profile,
            radius_meters=search_radius_meters
        )
    )

    # --------------------------------------------------------
    # DEBUG / VALIDATION
    # --------------------------------------------------------

    print("\n[DEBUG] Discovered node structure:")

    for key, value in discovered_nodes.items():

        print(
            f"  {key}: "
            f"{type(value).__name__}"
        )

        if isinstance(value, dict):
            print(
                f"      coordinates = "
                f"{value.get('coordinates')} "
                f"({type(value.get('coordinates')).__name__})"
            )

    # --------------------------------------------------------
    # STAGE 3
    # --------------------------------------------------------

    print(
        "\n[Stage 3] "
        "Generating vehicle specs and dispatch schedules..."
    )

    freight_specs = (
        generate_dynamic_freight_parameters(
            business_type=business_type,
            origin_name=origin_location,
            destinations=discovered_nodes,
            orig_lat=orig_lat,
            orig_lon=orig_lon
        )
    )

    # --------------------------------------------------------
    # STAGE 4
    # --------------------------------------------------------

    print(
        "\n[Stage 4] "
        "Calculating road distances & monthly freight OPEX..."
    )

    logistics_report = {}

    total_monthly_transport_opex = 0.0

    for node_key, node_data in discovered_nodes.items():

        if not isinstance(node_data, dict):
            raise TypeError(
                f"Node '{node_key}' is not a dictionary."
            )

        coordinates = node_data.get(
            "coordinates"
        )

        dest_lat, dest_lon = normalize_coordinates(
            coordinates
        )

        destination_name = node_data.get(
            "name",
            "Unknown Facility"
        )

        route_metrics = calculate_road_route(
            orig_lat,
            orig_lon,
            dest_lat,
            dest_lon
        )

        road_km = route_metrics[
            "road_distance_km"
        ]

        spec = freight_specs.get(
            node_key,
            {
                "recommended_vehicle":
                    "Mini Commercial Truck",
                "payload_capacity_kg":
                    1000,
                "base_fare_inr":
                    200.0,
                "base_km":
                    4.0,
                "rate_per_km_inr":
                    24.0,
                "round_trip_multiplier":
                    1.8,
                "monthly_trips":
                    4,
                "handling_notes":
                    "General freight"
            }
        )

        fares = compute_trip_cost(
            road_km,
            spec
        )

        monthly_trips = int(
            spec.get(
                "monthly_trips",
                4
            )
        )

        monthly_cost = round(
            fares["round_trip_inr"]
            * monthly_trips,
            2
        )

        total_monthly_transport_opex += (
            monthly_cost
        )

        logistics_report[node_key] = {
            "pillar_label":
                node_data.get(
                    "pillar_label"
                ),

            "status":
                node_data.get(
                    "status"
                ),

            "destination_facility":
                destination_name,

            "destination_coordinates": {
                "lat": dest_lat,
                "lon": dest_lon
            },

            "route_metrics":
                route_metrics,

            "vehicle_specification": {
                "vehicle":
                    spec.get(
                        "recommended_vehicle"
                    ),

                "payload_capacity_kg":
                    spec.get(
                        "payload_capacity_kg"
                    ),

                "handling_notes":
                    spec.get(
                        "handling_notes"
                    )
            },

            "freight_rates": {
                "one_way_fare_inr":
                    fares["one_way_inr"],

                "round_trip_fare_inr":
                    fares["round_trip_inr"],

                "rate_per_km_inr":
                    spec.get(
                        "rate_per_km_inr"
                    )
            },

            "dispatch_schedule": {
                "estimated_monthly_trips":
                    monthly_trips,

                "estimated_monthly_cost_inr":
                    monthly_cost
            }
        }

    # --------------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------------

    return {
        "business_type":
            business_type,

        "origin": {
            "name":
                origin_location,

            "coordinates": {
                "lat":
                    orig_lat,

                "lon":
                    orig_lon
            }
        },

        "discovered_supply_chain_nodes":
            discovered_nodes,

        "transportation_logistics_opex":
            logistics_report,

        "summary": {
            "active_routes_modeled":
                len(logistics_report),

            "total_monthly_freight_opex_inr":
                round(
                    total_monthly_transport_opex,
                    2
                )
        }
    }


# ============================================================
# EXECUTION DEMO
# ============================================================

if __name__ == "__main__":

    BUSINESS = "dairy"

    LOCATION = (
        "Katwa, Purba Bardhaman, "
        "West Bengal"
    )

    output = (
        run_supply_chain_and_transportation_analysis(
            business_type=BUSINESS,
            origin_location=LOCATION,
            search_radius_meters=30000
        )
    )

    print(
        "\n--- FINAL PIPELINE OUTPUT ---"
    )

    preview = {
        "business_type":
            output["business_type"],

        "origin":
            output["origin"],

        "summary":
            output["summary"],

        "routes": {
            key: {
                "pillar":
                    value["pillar_label"],

                "status":
                    value["status"],

                "facility":
                    value["destination_facility"],

                "road_km":
                    value["route_metrics"][
                        "road_distance_km"
                    ],

                "vehicle":
                    value["vehicle_specification"][
                        "vehicle"
                    ],

                "monthly_trips":
                    value["dispatch_schedule"][
                        "estimated_monthly_trips"
                    ],

                "monthly_opex_inr":
                    value["dispatch_schedule"][
                        "estimated_monthly_cost_inr"
                    ]
            }

            for key, value
            in output[
                "transportation_logistics_opex"
            ].items()
        }
    }

    print(
        json.dumps(
            preview,
            indent=2,
            ensure_ascii=False
        )
    )

INTEGRATED SUPPLY CHAIN & TRANSPORTATION OPEX PIPELINE
Business Profile : dairy
Origin Location  : Katwa, Purba Bardhaman, West Bengal
Origin Resolved  : 23.6443, 88.1287

[Stage 1] Formulating supply chain requirements...

[Stage 2] Scanning OpenStreetMap for physical supply chain hubs...
Searching Overpass for 'Milk Collection & Chilling Centers' within 30.0 km...
Searching Overpass for 'Feed & Agri-Input Outlets' within 30.0 km...
Searching Overpass for 'Veterinary & Cattle Healthcare' within 30.0 km...
[OVERPASS] HTTP 429
Searching Overpass for 'Rural Transport & Logistics Hubs' within 30.0 km...
[OVERPASS] HTTP 504

[DEBUG] Discovered node structure:
  milk_collection_and_chilling: dict
      coordinates = {'lat': 23.5813513, 'lon': 88.2279035} (dict)
  feed_and_agricultural_inputs: dict
      coordinates = {'lat': 23.5990814, 'lon': 88.394504} (dict)
  veterinary_and_health_services: dict
      coordinates = {'lat': 23.724277999999998, 'lon': 88.1887333} (dict)
  rural_logistics_

In [21]:
import os
import math
import json
import requests
from datetime import datetime, timedelta
from typing import Any, Dict, List, Tuple, Optional

from google import genai
from google.genai import types

try:
    from google.colab import userdata
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False


# ============================================================
# NUMERIC SAFETY HELPERS
# ============================================================

def safe_float(value: Any, default: float = 0.0) -> float:
    """
    Safely convert strings/numbers to float.

    Examples:
        "105.5" -> 105.5
        105 -> 105.0
        None -> default
        "abc" -> default
    """
    if value is None:
        return default

    if isinstance(value, bool):
        return float(value)

    try:
        if isinstance(value, str):
            value = value.strip().replace(",", "")
            if not value:
                return default

        return float(value)
    except (TypeError, ValueError):
        return default


def safe_int(value: Any, default: int = 0) -> int:
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return default


def normalize_index(value: Any, default: float = 100.0) -> float:
    """
    Normalize an AI-generated index to a float.

    Prevents errors such as:
        '105.5' - 100.0
    """
    return safe_float(value, default)


# ============================================================
# 1. API CLIENT & GEOLOCATION
# ============================================================

def get_gemini_client() -> genai.Client:
    key = None

    if COLAB_AVAILABLE:
        try:
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass

    key = key or os.environ.get("GEMINI_API_KEY")

    if not key:
        raise ValueError(
            "Missing 'GEMINI_API_KEY'. "
            "Set it in Colab Secrets or as an environment variable."
        )

    return genai.Client(api_key=key)


def get_coordinates(location_name: str) -> Tuple[float, float]:
    params = {
        "q": location_name,
        "format": "json",
        "limit": 1
    }

    res = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=HEADERS,
        timeout=15
    )

    res.raise_for_status()

    data = res.json()

    if not data:
        raise ValueError(
            f"Could not resolve coordinates for: '{location_name}'"
        )

    return (
        safe_float(data[0]["lat"]),
        safe_float(data[0]["lon"])
    )


# ============================================================
# 2. DISTANCE & ROUTING
# ============================================================

def haversine_distance_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float
) -> float:

    lat1 = safe_float(lat1)
    lon1 = safe_float(lon1)
    lat2 = safe_float(lat2)
    lon2 = safe_float(lon2)

    r = 6371.0

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )

    return round(
        2 * r * math.asin(math.sqrt(a)),
        2
    )


def get_dynamic_road_distance(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float
) -> Tuple[float, float]:

    lat1 = safe_float(lat1)
    lon1 = safe_float(lon1)
    lat2 = safe_float(lat2)
    lon2 = safe_float(lon2)

    straight_line_km = haversine_distance_km(
        lat1,
        lon1,
        lat2,
        lon2
    )

    try:
        url = (
            f"{OSRM_URL}/"
            f"{lon1},{lat1};{lon2},{lat2}"
            f"?overview=false"
        )

        res = requests.get(
            url,
            headers=HEADERS,
            timeout=6
        )

        if res.status_code == 200:
            routes = res.json().get("routes", [])

            if routes:
                road_km = round(
                    safe_float(routes[0].get("distance")) / 1000.0,
                    2
                )

                duration_min = round(
                    safe_float(routes[0].get("duration")) / 60.0,
                    1
                )

                return road_km, duration_min

    except Exception as e:
        print(f"OSRM routing failed: {e}")

    # Fallback estimation
    est_road_km = round(
        straight_line_km * 1.30,
        2
    )

    est_duration_min = round(
        (est_road_km / 25.0) * 60.0,
        1
    )

    return est_road_km, est_duration_min


def calculate_dynamic_decay_score(
    distance_km: float,
    max_ideal_km: float = 5.0,
    max_cutoff_km: float = 30.0
) -> int:

    distance_km = safe_float(distance_km)
    max_ideal_km = safe_float(max_ideal_km, 5.0)
    max_cutoff_km = safe_float(max_cutoff_km, 30.0)

    if max_cutoff_km <= max_ideal_km:
        max_cutoff_km = max_ideal_km + 1.0

    if distance_km <= 0:
        return 100

    if distance_km <= max_ideal_km:
        return int(
            100 - (distance_km / max_ideal_km) * 10
        )

    if distance_km >= max_cutoff_km:
        return 10

    decay = (
        (distance_km - max_ideal_km)
        / (max_cutoff_km - max_ideal_km)
    )

    return max(
        10,
        int(90 - (decay * 80))
    )


# ============================================================
# 3. CLIMATE OBSERVATION ENGINE
# ============================================================

def fetch_rolling_climate_data(
    lat: float,
    lon: float
) -> List[Dict[str, Any]]:

    lat = safe_float(lat)
    lon = safe_float(lon)

    end_date = (
        datetime.now().date()
        - timedelta(days=5)
    )

    start_date = (
        end_date
        - timedelta(days=365)
    )

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "daily": [
            "temperature_2m_max",
            "precipitation_sum"
        ],
        "timezone": "auto"
    }

    try:
        res = requests.get(
            OPEN_METEO_ARCHIVE_URL,
            params=params,
            headers=HEADERS,
            timeout=15
        )

        res.raise_for_status()

        daily = res.json().get("daily", {})

        dates = daily.get("time", [])
        max_temps = daily.get(
            "temperature_2m_max",
            []
        )
        precips = daily.get(
            "precipitation_sum",
            []
        )

        monthly_rain = {
            m: 0.0
            for m in range(1, 13)
        }

        monthly_temp = {
            m: []
            for m in range(1, 13)
        }

        for d_str, temp, rain in zip(
            dates,
            max_temps,
            precips
        ):
            try:
                month = int(
                    d_str.split("-")[1]
                )
            except (IndexError, ValueError):
                continue

            if temp is not None:
                temp_value = safe_float(
                    temp,
                    default=float("nan")
                )

                if not math.isnan(temp_value):
                    monthly_temp[month].append(
                        temp_value
                    )

            if rain is not None:
                monthly_rain[month] += safe_float(
                    rain
                )

        monthly_results = []

        for m in range(1, 13):

            temps = monthly_temp[m]

            avg_t = (
                round(
                    sum(temps) / len(temps),
                    1
                )
                if temps
                else 28.0
            )

            rain_tot = round(
                safe_float(monthly_rain[m]),
                1
            )

            heat_stress = (
                round(
                    max(
                        0.0,
                        (avg_t - 32.0) / 10.0
                    ),
                    2
                )
                if avg_t > 32.0
                else 0.0
            )

            monsoon_risk = (
                round(
                    min(
                        1.0,
                        rain_tot / 300.0
                    ),
                    2
                )
                if rain_tot > 150.0
                else 0.0
            )

            monthly_results.append({
                "month_index": m,
                "month": MONTH_NAMES[m - 1],
                "avg_max_temp_c": float(avg_t),
                "rainfall_mm": float(rain_tot),
                "heat_stress_factor": float(
                    heat_stress
                ),
                "monsoon_risk_factor": float(
                    monsoon_risk
                )
            })

        return monthly_results

    except Exception as e:

        print(
            f"Weather fetch failed: {e}"
        )

        return [
            {
                "month_index": m,
                "month": MONTH_NAMES[m - 1],
                "avg_max_temp_c": 30.0,
                "rainfall_mm": 50.0,
                "heat_stress_factor": 0.0,
                "monsoon_risk_factor": 0.0
            }
            for m in range(1, 13)
        ]


# ============================================================
# 4. SUPPLY CHAIN PROFILE
# ============================================================

def generate_supply_chain_profile(
    business_type: str
) -> Dict[str, Any]:

    client = get_gemini_client()

    prompt = f"""
You are an expert rural/peri-urban supply chain architect.

Target Business: "{business_type}"

Define exactly 3 critical upstream or downstream
infrastructure pillars required for this business.

Provide realistic Overpass QL tag selectors for each.

Return strict JSON:

{{
  "pillars": [
    {{
      "id": "pillar_identifier",
      "label": "Display Name",
      "weight": 0.35,
      "ideal_km": 8.0,
      "cutoff_km": 30.0,
      "osm_queries": [
        "nwr['amenity'='marketplace']",
        "nwr['shop'='wholesale']"
      ]
    }}
  ]
}}

Weights must sum to 1.0.
"""

    resp = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1
        )
    )

    profile = json.loads(resp.text)

    # Normalize numeric Gemini output
    pillars = profile.get("pillars", [])

    normalized_pillars = []

    for pillar in pillars:

        weight = safe_float(
            pillar.get("weight"),
            0.0
        )

        ideal_km = safe_float(
            pillar.get("ideal_km"),
            8.0
        )

        cutoff_km = safe_float(
            pillar.get("cutoff_km"),
            30.0
        )

        normalized_pillars.append({
            "id": str(
                pillar.get(
                    "id",
                    f"pillar_{len(normalized_pillars) + 1}"
                )
            ),
            "label": str(
                pillar.get(
                    "label",
                    "Infrastructure"
                )
            ),
            "weight": weight,
            "ideal_km": ideal_km,
            "cutoff_km": cutoff_km,
            "osm_queries": [
                str(q)
                for q in pillar.get(
                    "osm_queries",
                    []
                )
            ]
        })

    # Normalize weights
    total_weight = sum(
        p["weight"]
        for p in normalized_pillars
    )

    if total_weight > 0:
        for p in normalized_pillars:
            p["weight"] = (
                p["weight"]
                / total_weight
            )

    profile["pillars"] = normalized_pillars

    return profile


# ============================================================
# 5. OVERPASS SUPPLY CHAIN DISCOVERY
# ============================================================

def discover_supply_chain_nodes(
    lat: float,
    lon: float,
    profile: Dict[str, Any],
    radius_meters: int = 25000
) -> Dict[str, Any]:

    lat = safe_float(lat)
    lon = safe_float(lon)

    discovered = {}

    for pillar in profile.get(
        "pillars",
        []
    ):

        pid = str(
            pillar.get("id", "unknown")
        )

        label = str(
            pillar.get(
                "label",
                "Infrastructure"
            )
        )

        weight = safe_float(
            pillar.get("weight"),
            0.0
        )

        ideal_km = safe_float(
            pillar.get("ideal_km"),
            8.0
        )

        cutoff_km = safe_float(
            pillar.get("cutoff_km"),
            30.0
        )

        clauses = "\n".join(
            f"  {q}(around:{int(radius_meters)},{lat},{lon});"
            for q in pillar.get(
                "osm_queries",
                []
            )
        )

        query = (
            "[out:json][timeout:25];\n"
            "(\n"
            f"{clauses}\n"
            ");\n"
            "out center 10;"
        )

        elements = []

        try:
            res = requests.post(
                OVERPASS_URL,
                data={"data": query},
                headers=HEADERS,
                timeout=30
            )

            if res.status_code == 200:
                elements = res.json().get(
                    "elements",
                    []
                )
            else:
                print(
                    f"Overpass returned "
                    f"HTTP {res.status_code}"
                )

        except Exception as e:
            print(
                f"Overpass failed for "
                f"{label}: {e}"
            )

        candidates = []

        for el in elements:

            center = el.get(
                "center",
                {}
            )

            p_lat = (
                el.get("lat")
                if el.get("lat") is not None
                else center.get("lat")
            )

            p_lon = (
                el.get("lon")
                if el.get("lon") is not None
                else center.get("lon")
            )

            if p_lat is None or p_lon is None:
                continue

            p_lat = safe_float(p_lat)
            p_lon = safe_float(p_lon)

            tags = el.get("tags", {})

            name = (
                tags.get("name")
                or tags.get("operator")
                or f"Regional {label} Node"
            )

            candidates.append({
                "name": str(name),
                "lat": p_lat,
                "lon": p_lon
            })

        if candidates:

            nearest = min(
                candidates,
                key=lambda p:
                    haversine_distance_km(
                        lat,
                        lon,
                        p["lat"],
                        p["lon"]
                    )
            )

            dist_km = haversine_distance_km(
                lat,
                lon,
                nearest["lat"],
                nearest["lon"]
            )

            discovered[pid] = {
                "label": label,
                "name": nearest["name"],
                "coordinates": (
                    nearest["lat"],
                    nearest["lon"]
                ),
                "weight": weight,
                "ideal_km": ideal_km,
                "cutoff_km": cutoff_km,
                "status": "Found"
            }

        else:

            discovered[pid] = {
                "label": label,
                "name": (
                    f"Sub-Divisional "
                    f"{label} Hub (Estimated)"
                ),
                "coordinates": (
                    lat + 0.05,
                    lon + 0.05
                ),
                "weight": weight,
                "ideal_km": ideal_km,
                "cutoff_km": cutoff_km,
                "status": "Unmapped"
            }

    return discovered


# ============================================================
# 6. SEASONALITY & WEATHER RISK ENGINE
# ============================================================

def evaluate_dynamic_seasonality_and_risks(
    business_type: str,
    climate_profile: List[Dict[str, Any]],
    user_prices: Optional[List[float]] = None
) -> Dict[str, Any]:

    client = get_gemini_client()

    # --------------------------------------------------------
    # NORMALIZE CLIMATE DATA
    # --------------------------------------------------------

    weather_summary = []

    for c in climate_profile:

        weather_summary.append({
            "month": str(
                c.get("month", "")
            ),
            "temp_c": safe_float(
                c.get("avg_max_temp_c"),
                28.0
            ),
            "rain_mm": safe_float(
                c.get("rainfall_mm"),
                0.0
            ),
            "heat_stress": safe_float(
                c.get("heat_stress_factor"),
                0.0
            ),
            "monsoon_risk": safe_float(
                c.get("monsoon_risk_factor"),
                0.0
            )
        })

    # --------------------------------------------------------
    # NORMALIZE USER PRICES
    # --------------------------------------------------------

    normalized_prices = None

    if user_prices:

        normalized_prices = []

        for price in user_prices:

            value = safe_float(
                price,
                default=float("nan")
            )

            if not math.isnan(value):
                normalized_prices.append(value)

        if not normalized_prices:
            normalized_prices = None

    price_text = (
        json.dumps(normalized_prices)
        if normalized_prices
        else "None (estimate realistic Base-100 cycle)"
    )

    # --------------------------------------------------------
    # GEMINI PROMPT
    # --------------------------------------------------------

    prompt = f"""
You are an agricultural economist and business risk analyst.

Analyze the enterprise:
"{business_type}"

Observed 12-month meteorological conditions:

{json.dumps(weather_summary, indent=2)}

User-Supplied Mandi Prices:

{price_text}

Task:

For each of the 12 months compute:

1. price_index
   Base-100 price index.
   100 = annual average.

2. demand_index
   Base-100 consumer/buyer demand index.

3. production_index
   Base-100 output/throughput index considering:
   - heat
   - rainfall
   - harvest seasons
   - operational downtime
   - seasonal business conditions

Identify vulnerable lean months where revenue
or production experiences substantial stress.

Provide a concise moratorium advisory explaining
when the borrower may face cash-flow pressure.

IMPORTANT:
All index values MUST be JSON NUMBERS.
Do NOT return them as strings.

Example:

"price_index": 105.5

NOT:

"price_index": "105.5"

Return exactly 12 monthly records.

Output strictly valid JSON:

{{
  "monthly_profile": [
    {{
      "month": "Jan",
      "price_index": 100.0,
      "demand_index": 95.0,
      "production_index": 110.0
    }}
  ],
  "vulnerable_lean_months": ["Jun", "Jul"],
  "moratorium_advisory": "Strategic loan repayment timing guidance..."
}}
"""

    try:

        resp = client.models.generate_content(
            model=GEMINI_MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.1
            )
        )

        analysis = json.loads(resp.text)

    except Exception as e:

        raise RuntimeError(
            f"Gemini seasonality generation failed: {e}"
        )

    # --------------------------------------------------------
    # MERGE + FORCE ALL NUMERIC VALUES TO FLOAT
    # --------------------------------------------------------

    ai_months = analysis.get(
        "monthly_profile",
        []
    )

    merged_months = []

    for i, clim in enumerate(
        climate_profile
    ):

        if (
            i < len(ai_months)
            and isinstance(
                ai_months[i],
                dict
            )
        ):
            ai_data = ai_months[i]
        else:
            ai_data = {}

        # CRITICAL FIX:
        # Gemini may return "105.5" instead of 105.5.
        # safe_float() guarantees numeric output.

        price_index = normalize_index(
            ai_data.get(
                "price_index"
            ),
            100.0
        )

        demand_index = normalize_index(
            ai_data.get(
                "demand_index"
            ),
            100.0
        )

        production_index = normalize_index(
            ai_data.get(
                "production_index"
            ),
            100.0
        )

        avg_temp = safe_float(
            clim.get(
                "avg_max_temp_c"
            ),
            28.0
        )

        rainfall = safe_float(
            clim.get(
                "rainfall_mm"
            ),
            0.0
        )

        heat_stress = safe_float(
            clim.get(
                "heat_stress_factor"
            ),
            0.0
        )

        monsoon_risk = safe_float(
            clim.get(
                "monsoon_risk_factor"
            ),
            0.0
        )

        merged_months.append({
            "month": str(
                clim.get("month", "")
            ),
            "price_index": price_index,
            "demand_index": demand_index,
            "production_index": production_index,
            "avg_temp_c": avg_temp,
            "rainfall_mm": rainfall,
            "heat_stress_factor": heat_stress,
            "monsoon_risk_factor": monsoon_risk
        })

    # --------------------------------------------------------
    # NORMALIZE VULNERABLE MONTHS
    # --------------------------------------------------------

    vulnerable_months = analysis.get(
        "vulnerable_lean_months",
        []
    )

    if not isinstance(
        vulnerable_months,
        list
    ):
        vulnerable_months = []

    vulnerable_months = [
        str(month)
        for month in vulnerable_months
    ]

    advisory = str(
        analysis.get(
            "moratorium_advisory",
            ""
        )
    )

    return {
        "monthly_index_profile": merged_months,
        "vulnerable_lean_months": vulnerable_months,
        "moratorium_advisory": advisory
    }


# ============================================================
# 7. MASTER INTEGRATED ADVISORY ENGINE
# ============================================================

def generate_dynamic_advisory_report(
    location_name: str,
    business_type: str,
    sample_12m_mandi_prices: Optional[List[float]] = None
) -> Dict[str, Any]:

    print("=" * 65)
    print(
        "DYNAMIC BUSINESS INTELLIGENCE "
        "& RISK ADVISORY PIPELINE"
    )
    print("=" * 65)

    print(
        f"Business Profile : {business_type}"
    )

    print(
        f"Target Location  : {location_name}"
    )

    # --------------------------------------------------------
    # 1. GEOCODING
    # --------------------------------------------------------

    print("\n[1/5] Geocoding...")

    lat, lon = get_coordinates(
        location_name
    )

    print(
        f"Resolved Location: "
        f"{lat:.4f}, {lon:.4f}"
    )

    # --------------------------------------------------------
    # 2. CLIMATE
    # --------------------------------------------------------

    print(
        "\n[2/5] Fetching rolling "
        "12-month climate data..."
    )

    climate_profile = (
        fetch_rolling_climate_data(
            lat,
            lon
        )
    )

    print(
        f"Climate records: "
        f"{len(climate_profile)}"
    )

    # --------------------------------------------------------
    # 3. SUPPLY CHAIN PROFILE
    # --------------------------------------------------------

    print(
        "\n[3/5] Generating supply-chain profile..."
    )

    sc_profile = (
        generate_supply_chain_profile(
            business_type
        )
    )

    print(
        f"Pillars generated: "
        f"{len(sc_profile.get('pillars', []))}"
    )

    # --------------------------------------------------------
    # 4. OVERPASS DISCOVERY
    # --------------------------------------------------------

    print(
        "\n[4/5] Searching OpenStreetMap..."
    )

    discovered_nodes = (
        discover_supply_chain_nodes(
            lat,
            lon,
            sc_profile
        )
    )

    # --------------------------------------------------------
    # ROUTING + ACCESSIBILITY
    # --------------------------------------------------------

    sc_evaluation = {}
    weighted_sc_score = 0.0

    for pid, data in discovered_nodes.items():

        d_lat, d_lon = data[
            "coordinates"
        ]

        d_lat = safe_float(d_lat)
        d_lon = safe_float(d_lon)

        road_km, duration_min = (
            get_dynamic_road_distance(
                lat,
                lon,
                d_lat,
                d_lon
            )
        )

        accessibility_score = (
            calculate_dynamic_decay_score(
                distance_km=road_km,
                max_ideal_km=safe_float(
                    data.get("ideal_km"),
                    8.0
                ),
                max_cutoff_km=safe_float(
                    data.get("cutoff_km"),
                    30.0
                )
            )
        )

        weight = safe_float(
            data.get("weight"),
            0.0
        )

        weighted_sc_score += (
            accessibility_score
            * weight
        )

        sc_evaluation[pid] = {
            "pillar": str(
                data.get("label", "")
            ),
            "facility_name": str(
                data.get("name", "")
            ),
            "status": str(
                data.get("status", "")
            ),
            "road_distance_km": float(
                road_km
            ),
            "travel_time_minutes": float(
                duration_min
            ),
            "accessibility_score": (
                f"{accessibility_score}/100"
            )
        }

    # --------------------------------------------------------
    # 5. SEASONALITY
    # --------------------------------------------------------

    print(
        "\n[5/5] Synthesizing seasonal "
        "risks and cash-flow vulnerability..."
    )

    seasonality_assessment = (
        evaluate_dynamic_seasonality_and_risks(
            business_type=business_type,
            climate_profile=climate_profile,
            user_prices=sample_12m_mandi_prices
        )
    )

    # --------------------------------------------------------
    # FINAL REPORT
    # --------------------------------------------------------

    return {
        "location": location_name,

        "coordinates": {
            "lat": float(lat),
            "lon": float(lon)
        },

        "business_category": business_type,

        "supply_chain_metrics": {
            "overall_accessibility_score": (
                f"{int(round(weighted_sc_score))}/100"
            ),
            "nodes": sc_evaluation
        },

        "dynamic_seasonality_analysis": (
            seasonality_assessment
        )
    }


# ============================================================
# 8. RUN
# ============================================================

if __name__ == "__main__":

    BUSINESS = "cold storage facility"

    LOCATION = (
        "Katwa, Purba Bardhaman, West Bengal"
    )

    try:

        report = generate_dynamic_advisory_report(
            location_name=LOCATION,
            business_type=BUSINESS
        )

        print(
            "\n--- FINAL ADVISORY REPORT (PREVIEW) ---"
        )

        preview = {
            "business": report[
                "business_category"
            ],

            "location": report[
                "location"
            ],

            "overall_supply_chain_score": (
                report[
                    "supply_chain_metrics"
                ][
                    "overall_accessibility_score"
                ]
            ),

            "discovered_nodes": (
                report[
                    "supply_chain_metrics"
                ]["nodes"]
            ),

            "vulnerable_lean_months": (
                report[
                    "dynamic_seasonality_analysis"
                ][
                    "vulnerable_lean_months"
                ]
            ),

            "moratorium_advisory": (
                report[
                    "dynamic_seasonality_analysis"
                ][
                    "moratorium_advisory"
                ]
            )
        }

        print(
            json.dumps(
                preview,
                indent=2,
                ensure_ascii=False
            )
        )

    except Exception as e:

        print(f"\n❌ Pipeline failed: {e}")

DYNAMIC BUSINESS INTELLIGENCE & RISK ADVISORY PIPELINE
Business Profile : cold storage facility
Target Location  : Katwa, Purba Bardhaman, West Bengal

[1/5] Geocoding...
Resolved Location: 23.6443, 88.1287

[2/5] Fetching rolling 12-month climate data...
Climate records: 12

[3/5] Generating supply-chain profile...
Pillars generated: 3

[4/5] Searching OpenStreetMap...
Overpass failed for Agricultural Production Zones & Farms: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f6915a8df90>: Failed to establish a new connection: [Errno 101] Network is unreachable'))
Overpass failed for Major Highways & Freight Routes: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f6915a8f9d0>: Failed to establish a new connection: [Errno 

In [9]:
!pip install -U langchain langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled goog

In [10]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

In [22]:
import os
import json
from typing import Any
from IPython.display import display, Markdown
from langchain_google_genai import ChatGoogleGenerativeAI

# ============================================================
# IMPORT YOUR EXISTING DATA MODULES
# ============================================================

# from population import analyze_market_reach
# from competitor import analyze_competitors
# from market_price import analyze_market_price
# from supply_chain import evaluate_supply_chain
# from transportation import run_supply_chain_and_transportation_analysis
# from seasonality import generate_dynamic_advisory_report

# ============================================================
# HELPERS
# ============================================================

def print_tool_status(name: str, success: bool, message: str = ""):
    status = "✅ SUCCESS" if success else "❌ FAILED"
    print(f"\n[{name}] {status}")
    if message:
        print(f"[{name}] {message}")


def safe_json(data: Any):
    """Make sure data is JSON serializable."""
    try:
        json.dumps(data)
        return data
    except Exception:
        return str(data)


def run_module(name: str, function, **kwargs) -> dict:
    """
    Execute one data-analysis module safely.

    A module failure is represented explicitly as:
        status = failed
        data_available = false

    This is later passed to Gemini so that it knows
    the difference between UNKNOWN and zero.
    """

    print("\n" + "=" * 70)
    print(f"[{name}] STARTING")
    print("=" * 70)

    try:
        result = function(**kwargs)

        print_tool_status(
            name,
            True,
            "Module completed successfully."
        )

        return {
            "status": "success",
            "data_available": True,
            "data": safe_json(result),
            "error": None
        }

    except Exception as e:

        print_tool_status(
            name,
            False,
            str(e)
        )

        return {
            "status": "failed",
            "data_available": False,
            "data": None,
            "error": str(e)
        }


# ============================================================
# POPULATION
# ============================================================

def run_population(
    location: str,
    radius_km: float = 10.0,
    year: int = 2025
) -> dict:

    return run_module(
        "POPULATION",
        analyze_market_reach,
        location=location,
        radius_km=radius_km,
        year=year
    )


# ============================================================
# COMPETITOR
# ============================================================

def run_competitor(
    latitude: float,
    longitude: float,
    population: float | None,
    radius_km: float,
    business_type: str,
    gemini_api_key: str
) -> dict:

    return run_module(
        "COMPETITOR",
        analyze_competitors,
        latitude=latitude,
        longitude=longitude,
        radius_km=radius_km,
        population=population,
        business_type=business_type,
        gemini_api_key=gemini_api_key
    )


# ============================================================
# MARKET PRICE
# ============================================================

def run_market_price(
    business_type: str,
    state: str
) -> dict:

    return run_module(
        "MARKET PRICE",
        analyze_market_price,
        business_type=business_type,
        state=state
    )


# ============================================================
# SUPPLY CHAIN
# ============================================================

def run_supply_chain(
    location: str,
    business_type: str
) -> dict:

    return run_module(
        "SUPPLY CHAIN",
        evaluate_supply_chain,
        location_name=location,
        business_type=business_type
    )


# ============================================================
# TRANSPORTATION
# ============================================================

def run_transportation(
    origin: str,
    business_type: str
) -> dict:

    return run_module(
        "TRANSPORTATION",
        run_supply_chain_and_transportation_analysis,
        origin_location=origin,
        business_type=business_type,
        search_radius_meters=30000
    )


# ============================================================
# SEASONALITY
# ============================================================

def run_seasonality(
    location: str,
    business_type: str
) -> dict:

    return run_module(
        "SEASONALITY",
        generate_dynamic_advisory_report,
        location_name=location,
        business_type=business_type
    )


# ============================================================
# SYSTEM PROMPT FOR GEMINI
# ============================================================

SYSTEM_PROMPT = """
You are an AI-driven rural business feasibility analyst.

Your job is to analyze real-world data collected by a deterministic
data collection pipeline and generate a practical Hyper-Local
Business Feasibility Report for a rural micro-entrepreneur.

IMPORTANT:

The Python application has ALREADY executed all available
data-collection modules.

You MUST NOT attempt to call external tools or APIs.

Your job is ONLY to analyze the supplied evidence and produce
the final report.

============================================================
CRITICAL DATA RULES
============================================================

1. NEVER invent population, household, competitor, price,
   transportation, supply-chain, weather or geographic data.

2. NEVER treat a failed module as zero.

If:

status = "failed"

or:

data_available = false

then the corresponding information is UNKNOWN.

Example:

BAD:
"There are 0 competitors."

GOOD:
"Competitor data could not be retrieved, so competitor density
cannot be reliably established."

3. Distinguish clearly between:

OBSERVED DATA
- Values directly returned by the data modules.

CALCULATED VALUES
- Values mathematically derived from observed data.

ESTIMATES
- Values explicitly identified as estimates by the data source.

INTERPRETATION
- Your reasoning about what the data means.

RECOMMENDATIONS
- Actions suggested based on the evidence.

4. Never convert missing data into an estimated value unless the
input data itself explicitly provides that estimate.

5. If one or more modules failed, continue analyzing the modules
that succeeded.

6. Data failures MUST reduce confidence in the affected section.

7. Never claim guaranteed profitability or guaranteed business
success.

8. Never assume an estimated/fallback location represents an
actual business, facility or infrastructure.

9. Competitor count is UNKNOWN when competitor data failed.

10. Price is UNKNOWN when relevant price data failed.

11. Do not use unrelated price data as a substitute for missing
local prices.

12. If the data contains contradictory values, explicitly mention
the contradiction rather than silently selecting a value.

============================================================
ANALYSIS REQUIREMENTS
============================================================

Analyze the following areas.

1. MARKET REACH

Analyze when data is available:

- Population within the selected radius
- Estimated households
- Population density
- Immediate consumer base
- 5 km and 10 km market reach
- Distribution channels
- Geographic market coverage

Do not invent population numbers if the population module failed.

------------------------------------------------------------

2. LOCAL COMPETITION

Analyze:

- Competitor count
- Competitor density
- Competition within 2 km / 5 km / 10 km
- Organized vs local competitors
- Competitive intensity
- Geographic distribution of competitors

If competitor data failed, explicitly state that competitor density
is UNKNOWN.

------------------------------------------------------------

3. MARKET AND PRICING

Analyze:

- Available commodity prices
- Minimum
- Median
- Mean
- Maximum
- Price volatility
- Wholesale opportunity
- Retail opportunity
- Price positioning
- Purchasing-power implications

Be extremely careful with commodity/unit mismatches.

For example, do NOT compare a price per quintal directly with a
retail price per litre without explaining the difference.

------------------------------------------------------------

4. SUPPLY CHAIN

Analyze:

- Veterinary services
- Feed suppliers
- Agricultural suppliers
- Markets
- Milk collection points
- Transport infrastructure
- Distance to important facilities
- Supply-chain bottlenecks
- Supply-chain dependency
- Supply-chain risk

Do not interpret "not found" as proof that a facility does not
exist unless the underlying data explicitly supports that claim.

Instead say:

"No facility was identified by the queried data source."

------------------------------------------------------------

5. TRANSPORTATION

Analyze:

- Road distance
- Straight-line distance
- Travel time
- Vehicle suitability
- Transportation cost
- Route feasibility
- Monthly logistics implications

If a transportation field is missing, do not invent it.

------------------------------------------------------------

6. SEASONALITY

Analyze:

- Temperature
- Rainfall
- Monsoon
- Heat stress
- Production changes
- Demand changes
- Seasonal price changes
- Seasonal operating risks

Only discuss seasonal price changes if actual price data exists.

------------------------------------------------------------

7. OPPORTUNITY ANALYSIS

Identify evidence-supported:

- Underserved products
- Underserved services
- Market gaps
- Distribution opportunities
- Differentiation strategies

Do not claim a market gap as fact when the evidence does not
support it. Clearly label such statements as hypotheses.

------------------------------------------------------------

8. THREAT ANALYSIS

Consider:

- Supply-chain bottlenecks
- Seasonal fluctuations
- Competition
- Price volatility
- Transport dependency
- Single-buyer dependency
- Weather risks
- Data uncertainty
- Capital constraints

------------------------------------------------------------

9. SWOT

Provide:

Strengths
Weaknesses
Opportunities
Threats

Every important SWOT point should be traceable to the supplied data.

------------------------------------------------------------

10. FINAL RECOMMENDATION

Provide:

- Overall feasibility
- Main reasons
- Biggest risks
- Recommended business model
- Recommended target market
- Recommended pricing strategy
- Recommended next steps

The recommendation must consider the available margin capital.

Never state that the business is guaranteed to succeed.

============================================================
CONFIDENCE
============================================================

Use:

HIGH
MEDIUM
LOW

Confidence should reflect data quality.

Examples:

HIGH:
Multiple reliable modules successfully returned relevant data.

MEDIUM:
Data exists but is partially aggregated, estimated or indirect.

LOW:
Important modules failed or the evidence is incomplete.

============================================================
OUTPUT FORMAT
============================================================

# Hyper-Local Business Feasibility Report

## 1. Executive Summary

Include:

- Location
- Business type
- Available margin capital
- Market radius
- Overall feasibility
- Overall confidence
- Short evidence-based summary

## 2. Market Reach

Include:
- Observed data
- Calculated values
- Interpretation
- Confidence

## 3. Local Competition

Include:
- Observed competitor data
- Competitive intensity
- Limitations
- Confidence

## 4. Market & Pricing Analysis

Include:
- Available prices
- Price statistics
- Interpretation
- Pricing strategy
- Confidence

## 5. Supply Chain Analysis

Include:
- Available infrastructure
- Missing/unmapped infrastructure
- Risks
- Confidence

## 6. Transportation & Logistics

Include:
- Distances
- Travel time
- Vehicle considerations
- Logistics cost
- Confidence

## 7. Seasonality & Local Risks

Include:
- Weather
- Production
- Demand
- Seasonal risks
- Confidence

## 8. Opportunity Analysis

Include:
- Market opportunities
- Potential gaps
- Differentiation

## 9. Threat Analysis

Include:
- Operational threats
- Financial threats
- Market threats
- Data uncertainty

## 10. SWOT Analysis

Use:

### Strengths
### Weaknesses
### Opportunities
### Threats

## 11. Overall Feasibility

Include:

- HIGH / MEDIUM / LOW feasibility
- Main reasons
- Major uncertainties

## 12. Recommended Business Model

Include:

- Business model
- Target customers
- Pricing strategy
- Distribution strategy
- Operating strategy
- Capital considerations

## 13. Key Data Limitations

List every important failed, missing, estimated or indirect
data source.

Do not hide failures.

============================================================
FINAL PRINCIPLE
============================================================

Base conclusions on the supplied data.

Do not make up missing evidence.

A lack of evidence is NOT evidence of absence.

A failed API means UNKNOWN.

A successful API result does not automatically mean HIGH confidence;
consider the quality and relevance of the returned data.
"""


# ============================================================
# GEMINI MODEL
# ============================================================

# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise RuntimeError(
        "GOOGLE_API_KEY environment variable is not configured."
    )


llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL_NAME,
    google_api_key=GOOGLE_API_KEY
)


# ============================================================
# LOCATION HELPERS
# ============================================================

def extract_coordinates(population_result: dict):
    """
    Attempt to extract latitude/longitude from the population
    module result.

    Adjust the candidate keys according to your actual
    analyze_market_reach() output.
    """

    if population_result.get("status") != "success":
        return None, None

    data = population_result.get("data")

    if not isinstance(data, dict):
        return None, None

    latitude = (
        data.get("latitude")
        or data.get("lat")
    )

    longitude = (
        data.get("longitude")
        or data.get("lon")
        or data.get("lng")
    )

    # Check nested location objects if necessary
    if latitude is None or longitude is None:

        location_data = data.get("location")

        if isinstance(location_data, dict):

            latitude = (
                location_data.get("latitude")
                or location_data.get("lat")
            )

            longitude = (
                location_data.get("longitude")
                or location_data.get("lon")
                or location_data.get("lng")
            )

    return latitude, longitude


def extract_population(population_result: dict):
    """
    Extract population from the population module result.
    """

    if population_result.get("status") != "success":
        return None

    data = population_result.get("data")

    if not isinstance(data, dict):
        return None

    population = data.get("population")

    if population is None:
        population = data.get("total_population")

    return population


def extract_state(location: str) -> str:
    """
    Extract state from a standard Indian location string.

    Example:
        Katwa, Purba Bardhaman, West Bengal
        -> West Bengal

    If your application already receives state separately,
    use that instead.
    """

    parts = [p.strip() for p in location.split(",") if p.strip()]

    if len(parts) >= 1:
        return parts[-1]

    return location


# ============================================================
# MAIN REPORT GENERATOR
# ============================================================

def generate_module1_report(
    location: str,
    business_type: str,
    margin_capital: float,
    radius_km: float = 10.0
) -> str:

    print("\n")
    print("=" * 80)
    print("MODULE 1 - HYPER-LOCAL BUSINESS FEASIBILITY REPORT")
    print("=" * 80)
    print(f"Location       : {location}")
    print(f"Business Type  : {business_type}")
    print(f"Margin Capital : ₹{margin_capital:,.2f}")
    print(f"Radius         : {radius_km} km")
    print("=" * 80)

    # ========================================================
    # 1. POPULATION
    # ========================================================

    population_result = run_population(
        location=location,
        radius_km=radius_km,
        year=2025
    )

    # ========================================================
    # EXTRACT COORDINATES
    # ========================================================

    latitude, longitude = extract_coordinates(
        population_result
    )

    population = extract_population(
        population_result
    )

    print("\n[PIPELINE] Extracted coordinates:")
    print(f"[PIPELINE] Latitude : {latitude}")
    print(f"[PIPELINE] Longitude: {longitude}")
    print(f"[PIPELINE] Population: {population}")

    # ========================================================
    # 2. COMPETITOR
    # ========================================================

    if latitude is not None and longitude is not None:

        competitor_result = run_competitor(
            latitude=float(latitude),
            longitude=float(longitude),
            population=population,
            radius_km=radius_km,
            business_type=business_type,
            gemini_api_key=GOOGLE_API_KEY
        )

    else:

        print(
            "\n[COMPETITOR] ⚠️ Skipped because population "
            "module did not provide coordinates."
        )

        competitor_result = {
            "status": "failed",
            "data_available": False,
            "data": None,
            "error": "Coordinates unavailable from population module."
        }

    # ========================================================
    # 3. MARKET PRICE
    # ========================================================

    state = extract_state(location)

    market_price_result = run_market_price(
        business_type=business_type,
        state=state
    )

    # ========================================================
    # 4. SUPPLY CHAIN
    # ========================================================

    supply_chain_result = run_supply_chain(
        location=location,
        business_type=business_type
    )

    # ========================================================
    # 5. TRANSPORTATION
    # ========================================================

    transportation_result = run_transportation(
        origin=location,
        business_type=business_type
    )

    # ========================================================
    # 6. SEASONALITY
    # ========================================================

    seasonality_result = run_seasonality(
        location=location,
        business_type=business_type
    )

    # ========================================================
    # COLLECT ALL EVIDENCE
    # ========================================================

    all_data = {
        "input": {
            "location": location,
            "business_type": business_type,
            "margin_capital": margin_capital,
            "radius_km": radius_km
        },

        "population": population_result,

        "competitors": competitor_result,

        "market_price": market_price_result,

        "supply_chain": supply_chain_result,

        "transportation": transportation_result,

        "seasonality": seasonality_result
    }

    # ========================================================
    # PRINT PIPELINE SUMMARY
    # ========================================================

    print("\n")
    print("=" * 80)
    print("DATA COLLECTION SUMMARY")
    print("=" * 80)

    modules = {
        "Population": population_result,
        "Competitor": competitor_result,
        "Market Price": market_price_result,
        "Supply Chain": supply_chain_result,
        "Transportation": transportation_result,
        "Seasonality": seasonality_result
    }

    for name, result in modules.items():

        status = result.get("status")

        if status == "success":
            print(f"✅ {name}: SUCCESS")
        else:
            print(
                f"❌ {name}: FAILED - "
                f"{result.get('error')}"
            )

    # ========================================================
    # SERIALIZE DATA FOR GEMINI
    # ========================================================

    evidence_json = json.dumps(
        all_data,
        indent=2,
        ensure_ascii=False,
        default=str
    )

    # ========================================================
    # GEMINI PROMPT
    # ========================================================

    user_prompt = f"""
Generate the complete Hyper-Local Business Feasibility Report.

The data collection pipeline has already executed.

You must analyze ONLY the evidence supplied below.

============================================================
BUSINESS INPUT
============================================================

Location:
{location}

Business Type:
{business_type}

Available Margin Capital:
₹{margin_capital:,.2f}

Market Radius:
{radius_km} km

============================================================
COLLECTED REAL-WORLD EVIDENCE
============================================================

{evidence_json}

============================================================
IMPORTANT
============================================================

Some modules may have failed.

A failed module is UNKNOWN.

Do NOT invent replacement values.

Do NOT assume zero competitors when competitor data failed.

Do NOT create prices when price data failed.

Do NOT claim infrastructure exists merely because the location
is rural or urban.

Clearly distinguish:

- observed data
- calculated values
- estimates
- interpretations
- recommendations

Generate the complete report using the required format from
the system instructions.
"""

    # ========================================================
    # SEND EVERYTHING TO GEMINI
    # ========================================================

    print("\n")
    print("=" * 80)
    print("[GEMINI] ANALYZING COLLECTED EVIDENCE")
    print("=" * 80)

    try:

        response = llm.invoke(
            [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )
        response = []

    except Exception as e:

        print("\n" + "=" * 80)
        print("[GEMINI] ❌ REPORT GENERATION FAILED")
        print(f"[GEMINI] Error: {e}")
        print("=" * 80)

        raise

    # ========================================================
    # EXTRACT RESPONSE
    # ========================================================

    content = getattr(
        response,
        "content",
        None
    )

    if content is None:

        raise RuntimeError(
            "Gemini returned no content."
        )

    if isinstance(content, list):

        text_parts = []

        for item in content:

            if isinstance(item, dict):

                if item.get("type") == "text":
                    text_parts.append(
                        item.get("text", "")
                    )

            elif isinstance(item, str):

                text_parts.append(item)

        content = "\n".join(text_parts)

    # ========================================================
    # DISPLAY REPORT
    # ========================================================

    print("\n")
    print("=" * 80)
    print("FINAL REPORT")
    print("=" * 80)

    display(
        Markdown(content)
    )

    print("=" * 80)

    return content


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    report = generate_module1_report(
        location="Katwa, Purba Bardhaman, West Bengal",
        business_type="dairy",
        margin_capital=50000,
        radius_km=10
    )



MODULE 1 - HYPER-LOCAL BUSINESS FEASIBILITY REPORT
Location       : Katwa, Purba Bardhaman, West Bengal
Business Type  : dairy
Margin Capital : ₹50,000.00
Radius         : 10 km

[POPULATION] STARTING
1. Geocoding location: Katwa, Purba Bardhaman, West Bengal
   Coordinates: 23.644278, 88.1287333
2. Querying country metadata...
   Found India (IND)
3. Building 10 km radius buffer...
4. Calculating population via WorldPop...
Submitting WorldPop task (attempt 1/5)...
WorldPop HTTP status: 200
WorldPop status: progress
WorldPop status: success
5. Looking up UN average household size...
Using cached UN dataset (0 days old).

[POPULATION] ✅ SUCCESS
[POPULATION] Module completed successfully.

[PIPELINE] Extracted coordinates:
[PIPELINE] Latitude : 23.644278
[PIPELINE] Longitude: 88.1287333
[PIPELINE] Population: 398331

[COMPETITOR] STARTING
AI-POWERED OSM COMPETITOR PIPELINE
Target Query : 'dairy'
Center Point : 23.644278, 88.1287333
Radius       : 10 km

[AI Taxonomy Cache] Loaded taxon

RuntimeError: Gemini returned no content.

In [ ]:
import os
import logging
from typing import Optional

from google import genai
from google.genai import types
from google.colab import userdata
# from IPython.display import display, Markdown

logger = logging.getLogger(__name__)


SUPPORTED_LANGUAGES = {
    "en": "English",
    "hi": "Hindi",
    "bn": "Bengali",
    "ta": "Tamil",
    "te": "Telugu",
    "mr": "Marathi",
    "gu": "Gujarati",
    "kn": "Kannada",
    "ml": "Malayalam",
    "pa": "Punjabi",
    "or": "Odia",
    "as": "Assamese",
}


class TranslationError(Exception):
    """Raised when translation fails."""
    pass


class Translator:
    def __init__(
        self,
        api_key: Optional[str] = None,
        model: str = GEMINI_MODEL_NAME,
    ):
        self.api_key = api_key or userdata.get("GOOGLE_API_KEY")

        if not self.api_key:
            raise ValueError(
                "GEMINI_API_KEY environment variable is not set."
            )

        self.model = model
        self.client = genai.Client(api_key=self.api_key)

    def translate(
        self,
        text: str,
        target_language: str,
        source_language: str = "English",
    ) -> str:
        """
        Translate text from source_language to target_language.

        target_language can be either:
        - language code: 'bn', 'hi', 'ta'
        - language name: 'Bengali', 'Hindi', 'Tamil'
        """

        if not text or not text.strip():
            return text

        target_language_name = self._resolve_language(target_language)

        if (
            source_language.lower() == target_language_name.lower()
        ):
            return text

        prompt = self._build_prompt(
            text=text,
            source_language=source_language,
            target_language=target_language_name,
        )

        try:
            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    max_output_tokens=60000,
                ),
            )

            translated_text = response.text

            if not translated_text:
                raise TranslationError(
                    "Translation model returned an empty response."
                )

            return translated_text.strip()

        except Exception as exc:
            logger.exception("Translation failed")
            raise TranslationError(
                f"Translation failed: {exc}"
            ) from exc

    def _resolve_language(self, language: str) -> str:
        """
        Convert a language code into its full language name.
        """

        language = language.strip()

        # Language code
        if language.lower() in SUPPORTED_LANGUAGES:
            return SUPPORTED_LANGUAGES[language.lower()]

        # Language name
        for code, name in SUPPORTED_LANGUAGES.items():
            if language.lower() == name.lower():
                return name

        raise ValueError(
            f"Unsupported language: {language}. "
            f"Supported languages: "
            f"{', '.join(SUPPORTED_LANGUAGES.keys())}"
        )

    @staticmethod
    def _build_prompt(
        text: str,
        source_language: str,
        target_language: str,
    ) -> str:

        return f"""
You are a professional translation engine.

Translate the following text from {source_language} to {target_language}.

IMPORTANT RULES:

1. Translate the meaning accurately.
2. Do NOT add information.
3. Do NOT remove information.
4. Do NOT summarize the text.
5. Do NOT explain the translation.
6. Return ONLY the translated text.
7. Preserve all numbers exactly.
8. Preserve percentages exactly.
9. Preserve monetary values exactly.
10. Preserve dates exactly.
11. Preserve URLs exactly.
12. Preserve email addresses exactly.
13. Preserve IDs and identifiers exactly.
14. Preserve markdown formatting.
15. Preserve bullet points and numbered lists.
16. Preserve paragraph structure where possible.
17. Preserve government scheme names if they are official names.
18. Do not translate code, JSON keys, variable names, or technical identifiers.
19. Do not change factual information.
20. Do not add greetings, notes, or comments.

TEXT TO TRANSLATE:

{text}
"""


def translate_text(
    text: str,
    target_language: str,
) -> str:
    """
    Convenience function for simple usage.
    """

    translator = Translator()

    return translator.translate(
        text=text,
        target_language=target_language,
    )


if __name__ == "__main__":

    logging.basicConfig(level=logging.INFO)

    translator = Translator()

    translated_text = translator.translate(
        text=report,
        target_language="bn",
    )

    from IPython.display import display, Markdown

    print("\nTranslated text:\n")

    display(Markdown(translated_text))